In [ ]:
import pandas as pd

drug_nodes = pd.read_pickle("training_data/approved_small_molecule_drugs_review.pkl")
drugs_interactions_train = pd.read_pickle("training_data/drug_protein_interactions_train_review.pkl")
drugs_interactions_validation = pd.read_pickle("training_data/drug_protein_interactions_validation_review.pkl")
drugs_interactions_test = pd.read_pickle("training_data/drug_protein_interactions_test_review.pkl")
drug_indications = pd.read_pickle("training_data/drug_indications_review.pkl")
verified_negatives_protien_interactions_train = pd.read_pickle("training_data/verified_negatives_time_aware_train.pkl")
verified_negatives_protien_interactions_test = pd.read_pickle("training_data/verified_negatives_time_aware_test.pkl")
failed_indication_medium_negatives = pd.read_pickle("training_data/failed_indications_medium.pkl")
failed_indication_hard_negatives = pd.read_pickle("training_data/failed_indications_hard.pkl")
# protien nodes with embeddings
protein_nodes = pd.read_pickle("training_data/protein_nodes_with_embeddings_v4.pkl")

In [2]:
# convert SMILES to molecule object
from rdkit import Chem
import pandas as pd

print("="*80)
print("INVESTIGATING RDKIT MOLECULE OBJECT")
print("="*80)

# Get a sample drug from drug_nodes
sample_drug = drug_nodes.iloc[0]
print(f"\nSample Drug:")
print(f"  Name: {sample_drug['drug_name']}")
print(f"  ChEMBL ID: {sample_drug['drug_id']}")
print(f"  SMILES: {sample_drug['smile']}")

# Convert SMILES to molecule object
smiles = sample_drug['smile']
mol = Chem.MolFromSmiles(smiles)

print(f"\n{'='*80}")
print("MOLECULE OBJECT STRUCTURE")
print(f"{'='*80}")

# Basic info
print(f"\nBasic Properties:")
print(f"  Number of atoms: {mol.GetNumAtoms()}")
print(f"  Number of bonds: {mol.GetNumBonds()}")

# Investigate ATOMS
print(f"\n{'='*80}")
print("ATOMS (First 10)")
print(f"{'='*80}")

for i, atom in enumerate(mol.GetAtoms()):
    if i >= 10:
        print(f"  ... and {mol.GetNumAtoms() - 10} more atoms")
        break
    
    print(f"\nAtom {i}:")
    print(f"  Index:           {atom.GetIdx()}")
    print(f"  Symbol:          {atom.GetSymbol()}")  # C, N, O, etc.
    print(f"  Atomic Number:   {atom.GetAtomicNum()}")  # 6=C, 7=N, 8=O
    print(f"  Degree:          {atom.GetDegree()}")  # Number of bonds
    print(f"  Formal Charge:   {atom.GetFormalCharge()}")
    print(f"  Is Aromatic:     {atom.GetIsAromatic()}")
    print(f"  Hybridization:   {atom.GetHybridization()}")
    print(f"  Total H's:       {atom.GetTotalNumHs()}")
    print(f"  Valence:         {atom.GetTotalValence()}")

# Investigate BONDS
print(f"\n{'='*80}")
print("BONDS (First 10)")
print(f"{'='*80}")

for i, bond in enumerate(mol.GetBonds()):
    if i >= 10:
        print(f"  ... and {mol.GetNumBonds() - 10} more bonds")
        break
    
    atom1 = bond.GetBeginAtom()
    atom2 = bond.GetEndAtom()
    
    print(f"\nBond {i}:")
    print(f"  Connects:      Atom {bond.GetBeginAtomIdx()} ({atom1.GetSymbol()}) -- Atom {bond.GetEndAtomIdx()} ({atom2.GetSymbol()})")
    print(f"  Bond Type:     {bond.GetBondType()}")  # SINGLE, DOUBLE, TRIPLE, AROMATIC
    print(f"  Is Aromatic:   {bond.GetIsAromatic()}")
    print(f"  Is Conjugated: {bond.GetIsConjugated()}")

print("\n" + "="*80)
print("INVESTIGATION COMPLETE!")
print("="*80)
print("\n💡 What we learned:")
print("   - Each ATOM has: symbol (C, N, O...), index, degree, aromaticity")
print("   - Each BOND has: type (SINGLE, DOUBLE, TRIPLE, AROMATIC), connected atoms")
print("\nReady to proceed? Just let me know what you want to keep!")

INVESTIGATING RDKIT MOLECULE OBJECT

Sample Drug:
  Name: CETIRIZINE
  ChEMBL ID: CHEMBL1000
  SMILES: O=C(O)COCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1

MOLECULE OBJECT STRUCTURE

Basic Properties:
  Number of atoms: 27
  Number of bonds: 29

ATOMS (First 10)

Atom 0:
  Index:           0
  Symbol:          O
  Atomic Number:   8
  Degree:          1
  Formal Charge:   0
  Is Aromatic:     False
  Hybridization:   SP2
  Total H's:       0
  Valence:         2

Atom 1:
  Index:           1
  Symbol:          C
  Atomic Number:   6
  Degree:          3
  Formal Charge:   0
  Is Aromatic:     False
  Hybridization:   SP2
  Total H's:       0
  Valence:         4

Atom 2:
  Index:           2
  Symbol:          O
  Atomic Number:   8
  Degree:          1
  Formal Charge:   0
  Is Aromatic:     False
  Hybridization:   SP2
  Total H's:       1
  Valence:         2

Atom 3:
  Index:           3
  Symbol:          C
  Atomic Number:   6
  Degree:          2
  Formal Charge:   0
  Is Aromatic:     

In [3]:
from rdkit import Chem
import numpy as np
from tqdm import tqdm

print("="*80)
print("CONVERTING DRUG SMILES TO MOLECULAR GRAPHS")
print("="*80)

def smiles_to_graph(smiles):
    """
    Convert a SMILES string to a molecular graph.
    
    Returns:
        nodes: List of atom dictionaries with features
        edges: List of bond dictionaries (source, target, bond_type)
    """
    mol = Chem.MolFromSmiles(smiles)
    
    if mol is None:
        return None, None
    
    # Extract atom nodes
    nodes = []
    for atom in mol.GetAtoms():
        nodes.append({
            "id": atom.GetIdx(),
            "symbol": atom.GetSymbol(),
            "atomic_num": atom.GetAtomicNum(),
            "degree": atom.GetDegree(),
            "formal_charge": atom.GetFormalCharge(),
            "is_aromatic": atom.GetIsAromatic(),
            "hybridization": str(atom.GetHybridization()),
            "num_hs": atom.GetTotalNumHs()
        })
    
    # Extract bond edges (undirected)
    edges = []
    for bond in mol.GetBonds():
        bond_type = str(bond.GetBondType())
        edges.append({
            "source": bond.GetBeginAtomIdx(),
            "target": bond.GetEndAtomIdx(),
            "bond_type": bond_type
        })
    
    return nodes, edges

# Process all drugs
print(f"\nProcessing {len(drug_nodes)} drugs...")

drug_graphs = []
failed_drugs = []

for idx, row in tqdm(drug_nodes.iterrows(), total=len(drug_nodes), desc="Converting SMILES"):
    drug_internal_id = row['drug_internal_id']
    drug_id = row['drug_id']
    drug_name = row.get('drug_name', 'Unknown')
    smiles = row['smile']
    
    # Convert SMILES to graph
    nodes, edges = smiles_to_graph(smiles)
    
    if nodes is None:
        failed_drugs.append({
            'drug_internal_id': drug_internal_id,
            'drug_id': drug_id,
            'drug_name': drug_name,
            'smiles': smiles
        })
        continue
    
    drug_graphs.append({
        'drug_internal_id': drug_internal_id,
        'drug_id': drug_id,
        'drug_name': drug_name,
        'smiles': smiles,
        'num_atoms': len(nodes),
        'num_bonds': len(edges),
        'nodes': nodes,
        'edges': edges
    })

print(f"\n✓ Successfully processed: {len(drug_graphs)} drugs")
print(f"✗ Failed to parse: {len(failed_drugs)} drugs")

# Create summary statistics
print("\n" + "="*80)
print("MOLECULAR GRAPH STATISTICS")
print("="*80)

num_atoms_list = [d['num_atoms'] for d in drug_graphs]
num_bonds_list = [d['num_bonds'] for d in drug_graphs]

print(f"\nNumber of atoms per drug:")
print(f"  Min:    {min(num_atoms_list)}")
print(f"  Max:    {max(num_atoms_list)}")
print(f"  Mean:   {np.mean(num_atoms_list):.1f}")
print(f"  Median: {np.median(num_atoms_list):.1f}")

print(f"\nNumber of bonds per drug:")
print(f"  Min:    {min(num_bonds_list)}")
print(f"  Max:    {max(num_bonds_list)}")
print(f"  Mean:   {np.mean(num_bonds_list):.1f}")
print(f"  Median: {np.median(num_bonds_list):.1f}")

# Bond type distribution
bond_types = []
for d in drug_graphs:
    for edge in d['edges']:
        bond_types.append(edge['bond_type'])

bond_type_counts = pd.Series(bond_types).value_counts()
print(f"\nBond type distribution:")
for bond_type, count in bond_type_counts.items():
    print(f"  {bond_type}: {count:,} ({count/len(bond_types)*100:.1f}%)")

# Atom type distribution
atom_types = []
for d in drug_graphs:
    for node in d['nodes']:
        atom_types.append(node['symbol'])

atom_type_counts = pd.Series(atom_types).value_counts()
print(f"\nTop 10 most common atoms:")
for atom_type, count in atom_type_counts.head(10).items():
    print(f"  {atom_type}: {count:,} ({count/len(atom_types)*100:.1f}%)")

# Save results
print("\n" + "="*80)
print("SAVING RESULTS")
print("="*80)


# Example: Show first 3 drug graphs
print("\n" + "="*80)
print("EXAMPLE MOLECULAR GRAPHS (First 3 drugs)")
print("="*80)

for i, drug in enumerate(drug_graphs[:3], 1):
    print(f"\n{i}. {drug['drug_name']} ({drug['drug_id']})")
    print(f"   SMILES: {drug['smiles']}")
    print(f"   Atoms: {drug['num_atoms']}, Bonds: {drug['num_bonds']}")
    
    print(f"\n   Nodes (atoms):")
    for node in drug['nodes'][:5]:  # Show first 5 atoms
        print(f"     {node['id']}: {node['symbol']} (atomic_num={node['atomic_num']}, degree={node['degree']})")
    if len(drug['nodes']) > 5:
        print(f"     ... and {len(drug['nodes']) - 5} more atoms")
    
    print(f"\n   Edges (bonds):")
    for edge in drug['edges'][:5]:  # Show first 5 bonds
        print(f"     {edge['source']} -- {edge['target']} ({edge['bond_type']})")
    if len(drug['edges']) > 5:
        print(f"     ... and {len(drug['edges']) - 5} more bonds")

print("\n" + "="*80)
print("🎉 CONVERSION COMPLETE!")
print("="*80)

CONVERTING DRUG SMILES TO MOLECULAR GRAPHS

Processing 3127 drugs...


Converting SMILES:  76%|███████▌  | 2362/3127 [00:00<00:00, 4334.17it/s][11:51:43] WARNING: not removing hydrogen atom without neighbors
[11:51:43] WARNING: not removing hydrogen atom without neighbors
[11:51:43] WARNING: not removing hydrogen atom without neighbors
[11:51:43] WARNING: not removing hydrogen atom without neighbors
Converting SMILES: 100%|██████████| 3127/3127 [00:00<00:00, 4450.78it/s]


✓ Successfully processed: 3127 drugs
✗ Failed to parse: 0 drugs

MOLECULAR GRAPH STATISTICS

Number of atoms per drug:
  Min:    1
  Max:    200
  Mean:   27.6
  Median: 25.0

Number of bonds per drug:
  Min:    0
  Max:    210
  Mean:   29.0
  Median: 27.0

Bond type distribution:
  SINGLE: 53,356 (58.9%)
  AROMATIC: 30,161 (33.3%)
  DOUBLE: 6,908 (7.6%)
  TRIPLE: 152 (0.2%)

Top 10 most common atoms:
  C: 60,903 (70.6%)
  O: 13,321 (15.5%)
  N: 7,879 (9.1%)
  Cl: 1,106 (1.3%)
  S: 1,040 (1.2%)
  F: 1,028 (1.2%)
  Na: 273 (0.3%)
  I: 228 (0.3%)
  P: 158 (0.2%)
  Br: 101 (0.1%)

SAVING RESULTS

EXAMPLE MOLECULAR GRAPHS (First 3 drugs)

1. CETIRIZINE (CHEMBL1000)
   SMILES: O=C(O)COCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1
   Atoms: 27, Bonds: 29

   Nodes (atoms):
     0: O (atomic_num=8, degree=1)
     1: C (atomic_num=6, degree=3)
     2: O (atomic_num=8, degree=1)
     3: C (atomic_num=6, degree=2)
     4: O (atomic_num=8, degree=2)
     ... and 22 more atoms

   Edges (bonds):
     0 --

In [4]:
from collections import Counter
import pandas as pd

print("="*80)
print("COMPREHENSIVE ANALYSIS: ALL ATOM & BOND FEATURES")
print("="*80)

# Collect ALL features from atoms and bonds
atom_symbols = []
atom_degrees = []
atom_formal_charges = []
atom_is_aromatic = []
atom_hybridizations = []
atom_num_hs = []

bond_types = []

for drug in drug_graphs:
    # Collect ALL atom features
    for node in drug['nodes']:
        atom_symbols.append(node['symbol'])
        atom_degrees.append(node['degree'])
        atom_formal_charges.append(node['formal_charge'])
        atom_is_aromatic.append(node['is_aromatic'])
        atom_hybridizations.append(node['hybridization'])
        atom_num_hs.append(node['num_hs'])
    
    # Collect bond types
    for edge in drug['edges']:
        bond_types.append(edge['bond_type'])

# Count unique values
print("\n" + "="*80)
print("1️⃣  ATOM FEATURES")
print("="*80)

print(f"\n📊 Symbol (atom type):")
symbol_counts = Counter(atom_symbols)
print(f"   Total unique: {len(symbol_counts)}")
print(f"   Values: {sorted(symbol_counts.keys())}")
for symbol, count in sorted(symbol_counts.items(), key=lambda x: -x[1]):
    print(f"      {symbol:3s}: {count:6,} atoms ({count/len(atom_symbols)*100:.1f}%)")

print(f"\n📊 Degree (number of bonds):")
degree_counts = Counter(atom_degrees)
print(f"   Total unique: {len(degree_counts)}")
print(f"   Values: {sorted(degree_counts.keys())}")
for deg, count in sorted(degree_counts.items()):
    print(f"      {deg}: {count:,} atoms ({count/len(atom_degrees)*100:.1f}%)")

print(f"\n📊 Formal Charge:")
charge_counts = Counter(atom_formal_charges)
print(f"   Total unique: {len(charge_counts)}")
print(f"   Values: {sorted(charge_counts.keys())}")
for charge, count in sorted(charge_counts.items()):
    print(f"      {charge:+d}: {count:,} atoms ({count/len(atom_formal_charges)*100:.1f}%)")

print(f"\n📊 Is Aromatic:")
aromatic_counts = Counter(atom_is_aromatic)
print(f"   Total unique: {len(aromatic_counts)}")
for val, count in aromatic_counts.items():
    print(f"      {val}: {count:,} atoms ({count/len(atom_is_aromatic)*100:.1f}%)")

print(f"\n📊 Hybridization:")
hybrid_counts = Counter(atom_hybridizations)
print(f"   Total unique: {len(hybrid_counts)}")
print(f"   Values: {sorted(hybrid_counts.keys())}")
for hybrid, count in sorted(hybrid_counts.items(), key=lambda x: -x[1]):
    print(f"      {hybrid:10s}: {count:,} atoms ({count/len(atom_hybridizations)*100:.1f}%)")

print(f"\n📊 Number of Hydrogens:")
hs_counts = Counter(atom_num_hs)
print(f"   Total unique: {len(hs_counts)}")
print(f"   Values: {sorted(hs_counts.keys())}")
for hs, count in sorted(hs_counts.items()):
    print(f"      {hs}: {count:,} atoms ({count/len(atom_num_hs)*100:.1f}%)")

print("\n" + "="*80)
print("2️⃣  BOND FEATURES")
print("="*80)

print(f"\n📊 Bond Type:")
bond_counts = Counter(bond_types)
print(f"   Total unique: {len(bond_counts)}")
print(f"   Values: {sorted(bond_counts.keys())}")
for bond, count in sorted(bond_counts.items(), key=lambda x: -x[1]):
    print(f"      {bond:10s}: {count:,} bonds ({count/len(bond_types)*100:.1f}%)")

print("\n" + "="*80)
print("📋 SUMMARY FOR ONE-HOT ENCODING")
print("="*80)

print(f"\n✅ ATOM FEATURES TO ENCODE:")
print(f"   1. Symbol:         {len(symbol_counts):2d} unique values -> {len(symbol_counts):3d} dimensions")
print(f"   2. Degree:         {len(degree_counts):2d} unique values -> {len(degree_counts):3d} dimensions")
print(f"   3. Formal Charge:  {len(charge_counts):2d} unique values -> {len(charge_counts):3d} dimensions")
print(f"   4. Is Aromatic:    {len(aromatic_counts):2d} unique values -> {len(aromatic_counts):3d} dimensions")
print(f"   5. Hybridization:  {len(hybrid_counts):2d} unique values -> {len(hybrid_counts):3d} dimensions")
print(f"   6. Num Hydrogens:  {len(hs_counts):2d} unique values -> {len(hs_counts):3d} dimensions")

total_atom_features = (len(symbol_counts) + len(degree_counts) + 
                       len(charge_counts) + len(aromatic_counts) + 
                       len(hybrid_counts) + len(hs_counts))

print(f"\n   📊 Total ATOM embedding size: {total_atom_features} dimensions (one-hot)")

print(f"\n✅ BOND FEATURES TO ENCODE:")
print(f"   1. Bond Type:      {len(bond_counts):2d} unique values -> {len(bond_counts):3d} dimensions")

print(f"\n   📊 Total BOND embedding size: {len(bond_counts)} dimensions (one-hot)")

print("\n" + "="*80)
print("🎯 EMBEDDING BREAKDOWN")
print("="*80)
print(f"\nEach ATOM will be represented as a {total_atom_features}-dimensional vector")
print(f"Each BOND will be represented as a {len(bond_counts)}-dimensional vector")

print(f"\nExample for one drug molecule:")
sample_drug = drug_graphs[0]
print(f"  Drug: {sample_drug['drug_name']}")
print(f"  Atoms: {sample_drug['num_atoms']} x {total_atom_features} dims = {sample_drug['num_atoms'] * total_atom_features:,} values")
print(f"  Bonds: {sample_drug['num_bonds']} x {len(bond_counts)} dims = {sample_drug['num_bonds'] * len(bond_counts):,} values")

print("\n" + "="*80)
print("🎯 READY FOR ONE-HOT ENCODING!")
print("="*80)

COMPREHENSIVE ANALYSIS: ALL ATOM & BOND FEATURES

1️⃣  ATOM FEATURES

📊 Symbol (atom type):
   Total unique: 32
   Values: ['Ag', 'Al', 'As', 'B', 'Ba', 'Bi', 'Br', 'C', 'Ca', 'Cl', 'F', 'Ga', 'H', 'He', 'I', 'K', 'Kr', 'Li', 'Mg', 'N', 'Na', 'O', 'P', 'Ra', 'Rb', 'S', 'Se', 'Si', 'Sr', 'Xe', 'Yb', 'Zn']
      C  : 60,903 atoms (70.6%)
      O  : 13,321 atoms (15.5%)
      N  :  7,879 atoms (9.1%)
      Cl :  1,106 atoms (1.3%)
      S  :  1,040 atoms (1.2%)
      F  :  1,028 atoms (1.2%)
      Na :    273 atoms (0.3%)
      I  :    228 atoms (0.3%)
      P  :    158 atoms (0.2%)
      Br :    101 atoms (0.1%)
      K  :     35 atoms (0.0%)
      H  :     30 atoms (0.0%)
      Ca :     25 atoms (0.0%)
      Mg :     22 atoms (0.0%)
      Si :     14 atoms (0.0%)
      Li :      8 atoms (0.0%)
      Zn :      7 atoms (0.0%)
      Sr :      5 atoms (0.0%)
      B  :      5 atoms (0.0%)
      Se :      4 atoms (0.0%)
      Xe :      4 atoms (0.0%)
      As :      3 atoms (0.0%)
      Al :

In [5]:
# Create feature mappings for one-hot encoding
atom_feature_mappings = {
    'symbol': {val: i for i, val in enumerate(sorted(symbol_counts.keys()))},
    'degree': {val: i for i, val in enumerate(sorted(degree_counts.keys()))},
    'formal_charge': {val: i for i, val in enumerate(sorted(charge_counts.keys()))},
    'is_aromatic': {val: i for i, val in enumerate(sorted(aromatic_counts.keys()))},
    'hybridization': {val: i for i, val in enumerate(sorted(hybrid_counts.keys()))},
    'num_hs': {val: i for i, val in enumerate(sorted(hs_counts.keys()))}
}

bond_feature_mappings = {
    'bond_type': {val: i for i, val in enumerate(sorted(bond_counts.keys()))}
}

# Calculate total dimensions
total_atom_dim = sum(len(mapping) for mapping in atom_feature_mappings.values())
total_bond_dim = len(bond_feature_mappings['bond_type'])

print(f"\n✅ Feature mappings created:")
print(f"   Total atom dimensions: {total_atom_dim}")
print(f"   Total bond dimensions: {total_bond_dim}")

def encode_atom(atom_node):
    """Convert atom features to one-hot encoded vector"""
    vector = np.zeros(total_atom_dim)
    offset = 0
    for feature_name, mapping in atom_feature_mappings.items():
        value = atom_node[feature_name]
        if value in mapping:
            idx = offset + mapping[value]
            vector[idx] = 1.0
        offset += len(mapping)
    return vector

def encode_bond(bond_edge):
    """Convert bond features to one-hot encoded vector"""
    vector = np.zeros(total_bond_dim)
    bond_type = bond_edge['bond_type']
    if bond_type in bond_feature_mappings['bond_type']:
        idx = bond_feature_mappings['bond_type'][bond_type]
        vector[idx] = 1.0
    return vector

# Test encoding
sample_drug = drug_graphs[0]
first_atom = sample_drug['nodes'][0]
first_bond = sample_drug['edges'][0]

encoded_atom = encode_atom(first_atom)
encoded_bond = encode_bond(first_bond)

print(f"\n✅ Test encoding successful!")
print(f"   Atom vector shape: {encoded_atom.shape}")
print(f"   Bond vector shape: {encoded_bond.shape}")

# Encode all drugs
for drug in drug_graphs:
    # Node features matrix
    if drug['nodes']:
        node_attr = np.vstack([encode_atom(n) for n in drug['nodes']])
    else:
        node_attr = np.zeros((0, total_atom_dim), dtype=float)

    # Bond features and connectivity
    if drug['edges']:
        bond_features = np.vstack([encode_bond(e) for e in drug['edges']])
        sources = [int(e['source']) for e in drug['edges']]
        targets = [int(e['target']) for e in drug['edges']]

        # Bidirectional edges
        edge_index = np.array([sources + targets, targets + sources], dtype=np.int64)
        edge_attr = np.vstack([bond_features, bond_features])
    else:
        edge_index = np.zeros((2, 0), dtype=np.int64)
        edge_attr = np.zeros((0, total_bond_dim), dtype=float)

    # Save to drug dict
    drug['node_attr'] = node_attr
    drug['edge_attr'] = edge_attr
    drug['edge_index'] = edge_index

# Example output
sample = drug_graphs[0]
print("\n" + "="*80)
print(f"EXAMPLE: {sample['drug_name']}")
print("="*80)
print(f"  node_attr shape:  {sample['node_attr'].shape}")
print(f"  edge_attr shape:  {sample['edge_attr'].shape}")
print(f"  edge_index shape: {sample['edge_index'].shape}")
print("="*80)


✅ Feature mappings created:
   Total atom dimensions: 57
   Total bond dimensions: 4

✅ Test encoding successful!
   Atom vector shape: (57,)
   Bond vector shape: (4,)

EXAMPLE: CETIRIZINE
  node_attr shape:  (27, 57)
  edge_attr shape:  (58, 4)
  edge_index shape: (2, 58)


In [6]:
# Encode all drugs
for drug in drug_graphs:
    drug['encoded_atoms'] = [encode_atom(node) for node in drug['nodes']]
    drug['encoded_bonds'] = [encode_bond(edge) for edge in drug['edges']]

In [7]:
drug_graph_embeddings = []

for drug in drug_graphs:
    clean_drug = {
        # Identifiers (for tracking)
        'drug_internal_id': drug['drug_internal_id'],
        'drug_id': drug['drug_id'],
        'drug_name': drug['drug_name'],
        
        # Graph structure - ALL the model needs
        'node_attr': drug['node_attr'],      # (num_atoms, 57)
        'edge_attr': drug['edge_attr'],      # (num_edges*2, 4)
        'edge_index': drug['edge_index']     # (2, num_edges*2)
    }
    
    drug_graph_embeddings.append(clean_drug)

print(f"\n✓ Created {len(drug_graph_embeddings)} minimal drug embeddings")

# Show example
drug_graph_embeddings


✓ Created 3127 minimal drug embeddings


[{'drug_internal_id': 111185,
  'drug_id': 'CHEMBL1000',
  'drug_name': 'CETIRIZINE',
  'node_attr': array([[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 1., 0., 0.],
         ...,
         [0., 0., 0., ..., 1., 0., 0.],
         [0., 0., 0., ..., 0., 1., 0.],
         [0., 0., 0., ..., 0., 1., 0.]]),
  'edge_attr': array([[0., 1., 0., 0.],
         [0., 0., 1., 0.],
         [0., 0., 1., 0.],
         [0., 0., 1., 0.],
         [0., 0., 1., 0.],
         [0., 0., 1., 0.],
         [0., 0., 1., 0.],
         [0., 0., 1., 0.],
         [0., 0., 1., 0.],
         [0., 0., 1., 0.],
         [0., 0., 1., 0.],
         [0., 0., 1., 0.],
         [1., 0., 0., 0.],
         [1., 0., 0., 0.],
         [1., 0., 0., 0.],
         [1., 0., 0., 0.],
         [1., 0., 0., 0.],
         [0., 0., 1., 0.],
         [1., 0., 0., 0.],
         [1., 0., 0., 0.],
         [1., 0., 0., 0.],
         [0., 0., 1., 0.],
         [1., 0., 0., 0.],
         [1

In [8]:
import torch
from torch_geometric.data import Data
from tqdm import tqdm

print("="*80)
print("CONVERTING DRUGS TO PYTORCH GEOMETRIC DATA OBJECTS")
print("="*80)

# Convert each drug to a PyG Data object
drug_pyg_objects = []

# Find the cell where you create drug_pyg_objects (around line 1730)
# REPLACE IT WITH THIS:

drug_pyg_objects = []
failed_drugs = []

for drug in tqdm(drug_graph_embeddings, desc="Creating PyG objects"):
    # Validate BEFORE converting to tensors
    node_attr = drug['node_attr']
    edge_attr = drug['edge_attr']
    edge_index = drug['edge_index']
    
    # Check for NaN/Inf
    if np.isnan(node_attr).any() or np.isinf(node_attr).any():
        failed_drugs.append((drug['drug_id'], 'NaN/Inf in node_attr'))
        continue
    
    if np.isnan(edge_attr).any() or np.isinf(edge_attr).any():
        failed_drugs.append((drug['drug_id'], 'NaN/Inf in edge_attr'))
        continue
    
    # Check edge indices are valid
    num_atoms = node_attr.shape[0]
    if (edge_index >= num_atoms).any() or (edge_index < 0).any():
        failed_drugs.append((drug['drug_id'], f'Invalid edge_index (max={edge_index.max()}, num_atoms={num_atoms})'))
        continue
    
    # Create PyG Data object (ON CPU)
    x = torch.FloatTensor(node_attr)
    edge_index = torch.LongTensor(edge_index)
    edge_attr = torch.FloatTensor(edge_attr)
    
    data = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
        drug_internal_id=drug['drug_internal_id'],
        drug_id=drug['drug_id'],
        drug_name=drug['drug_name']
    )
    
    drug_pyg_objects.append(data)

print(f"\n✓ Created {len(drug_pyg_objects)} valid PyG objects")
print(f"✗ Failed: {len(failed_drugs)} drugs")

if failed_drugs:
    print("\nFailed drugs:")
    for drug_id, reason in failed_drugs[:10]:
        print(f"  {drug_id}: {reason}")

# Show example
print("\n" + "="*80)
print(f"EXAMPLE: {drug_pyg_objects[0].drug_name}")
print("="*80)
print(f"  drug_internal_id: {drug_pyg_objects[0].drug_internal_id}")
print(f"  drug_id:          {drug_pyg_objects[0].drug_id}")
print(f"  drug_name:        {drug_pyg_objects[0].drug_name}")
print(f"  x (node features): {drug_pyg_objects[0].x.shape}")
print(f"  edge_index:        {drug_pyg_objects[0].edge_index.shape}")
print(f"  edge_attr:         {drug_pyg_objects[0].edge_attr.shape}")
print(f"  num_nodes:         {drug_pyg_objects[0].num_nodes}")
print(f"  num_edges:         {drug_pyg_objects[0].num_edges}")

print("\n💾 Sample node features (first atom):")
print(drug_pyg_objects[0].x[0])  # one-hot encoded features

print("\n💾 Sample edge_index (first 5 edges):")
print(drug_pyg_objects[0].edge_index[:, :5])

print("\n" + "="*80)
print("✅ READY FOR GNN MODELING!")
print("="*80)
print("\nEach drug is now a PyG Data object with:")
print("  • x:          node features (atom embeddings)")
print("  • edge_index: graph connectivity")
print("  • edge_attr:  edge features (bond types)")
print("  • metadata:   drug_internal_id, drug_id, drug_name")
print("\n🎯 Next: Build GNN model to encode these into embeddings!")

/home/joe/projects/pharmacology-graph/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CONVERTING DRUGS TO PYTORCH GEOMETRIC DATA OBJECTS


Creating PyG objects: 100%|██████████| 3127/3127 [00:00<00:00, 26226.14it/s]


✓ Created 3127 valid PyG objects
✗ Failed: 0 drugs

EXAMPLE: CETIRIZINE
  drug_internal_id: 111185
  drug_id:          CHEMBL1000
  drug_name:        CETIRIZINE
  x (node features): torch.Size([27, 57])
  edge_index:        torch.Size([2, 58])
  edge_attr:         torch.Size([58, 4])
  num_nodes:         27
  num_edges:         58

💾 Sample node features (first atom):
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
        0., 0., 0., 0., 1., 0., 0., 0., 1., 0., 0., 0., 1., 0., 0., 0., 0., 1.,
        0., 0., 0.])

💾 Sample edge_index (first 5 edges):
tensor([[0, 1, 1, 3, 4],
        [1, 2, 3, 4, 5]])

✅ READY FOR GNN MODELING!

Each drug is now a PyG Data object with:
  • x:          node features (atom embeddings)
  • edge_index: graph connectivity
  • edge_attr:  edge features (bond types)
  • metadata:   drug_internal_id, drug_id, drug_name

🎯 Next: Build GNN model to en

In [9]:

print("="*80)
print("CONVERTING PROTEINS TO PYTORCH GEOMETRIC DATA OBJECTS")
print("="*80)

# Convert each protein to a PyG Data object
protein_pyg_objects = []

for idx, row in tqdm(protein_nodes.iterrows(), total=len(protein_nodes), desc="Creating PyG objects"):
    embedding_str = row['esm2_embedding']
    embedding = np.array(embedding_str)
    x = torch.FloatTensor(embedding).unsqueeze(0)  # Shape: (1, 1280)
    data = Data(
        x=x,  # node features (just one node per protein)
        protein_internal_id=int(row['protein_internal_id']),
        protein_id=str(row['protein_id']),
        protein_name=str(row['protein_name']),
        uniprot_id=str(row['uniprot_id']),
        sequence_length=int(row['sequence_length'])
    )
    
    protein_pyg_objects.append(data)

print(f"\n✓ Created {len(protein_pyg_objects)} PyTorch Geometric Data objects")

# Show example
print("\n" + "="*80)
print(f"EXAMPLE: {protein_pyg_objects[0].protein_name}")
print("="*80)
print(f"  protein_internal_id: {protein_pyg_objects[0].protein_internal_id}")
print(f"  protein_id:          {protein_pyg_objects[0].protein_id}")
print(f"  protein_name:        {protein_pyg_objects[0].protein_name}")
print(f"  uniprot_id:          {protein_pyg_objects[0].uniprot_id}")
print(f"  sequence_length:     {protein_pyg_objects[0].sequence_length}")
print(f"  x (embedding):       {protein_pyg_objects[0].x.shape}")

print("\n💾 Sample embedding (first 10 values):")
print(protein_pyg_objects[0].x[0, :10])



CONVERTING PROTEINS TO PYTORCH GEOMETRIC DATA OBJECTS


Creating PyG objects: 100%|██████████| 2053/2053 [00:00<00:00, 7637.53it/s]


✓ Created 2053 PyTorch Geometric Data objects

EXAMPLE: Maltase-glucoamylase
  protein_internal_id: 1
  protein_id:          CHEMBL2074
  protein_name:        Maltase-glucoamylase
  uniprot_id:          O43451
  sequence_length:     2753
  x (embedding):       torch.Size([1, 2560])

💾 Sample embedding (first 10 values):
tensor([ 0.0474,  0.0507, -0.0052, -0.0766, -0.0222, -0.0937, -0.0363,  0.0933,
         0.0464, -0.0282])


In [10]:
drug_effects = drug_indications

In [11]:
# Get unique effects from drug_effects
unique_effects = drug_effects[['effect_id', 'effect_name']].drop_duplicates()
print(f"\nTotal unique effects: {len(unique_effects)}")


Total unique effects: 1065


In [12]:

# Create simple embeddings for effects (we'll use one-hot encoding or learned embeddings later)
# For now, create random embeddings as placeholders
EFFECT_EMBEDDING_DIM = 32 # change if number of effects is bigger

effect_pyg_objects = []

for idx, row in tqdm(unique_effects.iterrows(), total=len(unique_effects), desc="Creating effect PyG objects"):
    effect_embedding = torch.randn(1, EFFECT_EMBEDDING_DIM)
    
    data = Data(
        x=effect_embedding,  # (1, 32)
        effect_id=str(row['effect_id']),
        effect_name=str(row['effect_name'])
    )
    
    effect_pyg_objects.append(data)

print(f"\n✓ Created {len(effect_pyg_objects)} effect PyG Data objects")

# Show example
print("\n" + "="*80)
print(f"EXAMPLE: {effect_pyg_objects[0].effect_name}")
print("="*80)
print(f"  effect_id:      {effect_pyg_objects[0].effect_id}")
print(f"  effect_name:    {effect_pyg_objects[0].effect_name}")
print(f"  x (embedding):  {effect_pyg_objects[0].x.shape}")

print("\n💾 Sample embedding (first 10 values):")
print(effect_pyg_objects[0].x[0, :10])

print("\n" + "="*80)
print("CREATING DRUG-EFFECT EDGE INDEX")
print("="*80)

# Create mappings: drug_internal_id -> index in drug_pyg_objects
drug_internal_id_to_idx = {}
for i, drug_data in enumerate(drug_pyg_objects):
    drug_internal_id_to_idx[drug_data.drug_internal_id] = i

# Create mappings: effect_id -> index in effect_pyg_objects
effect_id_to_idx = {}
for i, effect_data in enumerate(effect_pyg_objects):
    effect_id_to_idx[effect_data.effect_id] = i

print(f"\nDrug mapping: {len(drug_internal_id_to_idx)} drugs")
print(f"Effect mapping: {len(effect_id_to_idx)} effects")

# Create edge list: drug -> effect
drug_effect_edges = []
edge_attributes = []

for _, row in tqdm(drug_effects.iterrows(), total=len(drug_effects), desc="Creating drug-effect edges"):
    drug_internal_id = int(row['drug_internal_id'])
    effect_id = str(row['effect_id'])
    
    # Check if both exist in our mappings
    if drug_internal_id in drug_internal_id_to_idx and effect_id in effect_id_to_idx:
        drug_idx = drug_internal_id_to_idx[drug_internal_id]
        effect_idx = effect_id_to_idx[effect_id]
        
        # Add edge: drug -> effect
        drug_effect_edges.append([drug_idx, effect_idx])
        
        # Store edge attributes (indication_phase)
        edge_attributes.append({
            'indication_phase': float(row.get('indication_phase', 4.0)),
            'num_references': int(row.get('num_references', 1))
        })

# Convert to tensors
drug_effect_edge_index = torch.LongTensor(drug_effect_edges).t().contiguous()  # Shape: (2, num_edges)

print(f"\n✓ Created {drug_effect_edge_index.shape[1]} drug-effect edges")
print(f"  Edge index shape: {drug_effect_edge_index.shape}")

# Create edge attribute tensor
indication_phases = torch.FloatTensor([attr['indication_phase'] for attr in edge_attributes])
num_references = torch.FloatTensor([attr['num_references'] for attr in edge_attributes])

drug_effect_edge_attr = torch.stack([indication_phases, num_references], dim=1)  # Shape: (num_edges, 2)
print(f"  Edge attr shape: {drug_effect_edge_attr.shape}")

print("\n💾 Sample edges (first 5):")
print(f"  Edge index:\n{drug_effect_edge_index[:, :5]}")
print(f"  Edge attributes:\n{drug_effect_edge_attr[:5]}")

print("\n" + "="*80)
print("✅ DRUG-EFFECT GRAPH READY!")
print("="*80)
print("\nSummary:")
print(f"  • {len(drug_pyg_objects)} drug nodes (with molecular graphs)")
print(f"  • {len(effect_pyg_objects)} effect nodes (with embeddings)")
print(f"  • {drug_effect_edge_index.shape[1]} drug→effect edges")



Creating effect PyG objects: 100%|██████████| 1065/1065 [00:00<00:00, 44141.41it/s]



✓ Created 1065 effect PyG Data objects

EXAMPLE: Eye Manifestations
  effect_id:      D005132
  effect_name:    Eye Manifestations
  x (embedding):  torch.Size([1, 32])

💾 Sample embedding (first 10 values):
tensor([-0.5525,  1.7011,  0.8889, -0.7138, -0.5734, -2.7955, -0.9579, -0.7383,
         0.6822, -1.1532])

CREATING DRUG-EFFECT EDGE INDEX

Drug mapping: 3127 drugs
Effect mapping: 1065 effects


Creating drug-effect edges: 100%|██████████| 7086/7086 [00:00<00:00, 73468.78it/s]


✓ Created 5633 drug-effect edges
  Edge index shape: torch.Size([2, 5633])
  Edge attr shape: torch.Size([5633, 2])

💾 Sample edges (first 5):
  Edge index:
tensor([[0, 0, 1, 3, 3],
        [0, 1, 2, 3, 4]])
  Edge attributes:
tensor([[4., 1.],
        [4., 1.],
        [4., 1.],
        [4., 1.],
        [4., 1.]])

✅ DRUG-EFFECT GRAPH READY!

Summary:
  • 3127 drug nodes (with molecular graphs)
  • 1065 effect nodes (with embeddings)
  • 5633 drug→effect edges


In [13]:
print("="*80)
print("CREATING COMPREHENSIVE NODE & EDGE MAPPINGS")
print("="*80)

# Save all mappings for future use
graph_data = {
    # Nodes
    'drug_pyg_objects': drug_pyg_objects,
    'effect_pyg_objects': effect_pyg_objects,
    'protein_pyg_objects': protein_pyg_objects,  # Already created
    
    # Node mappings
    'drug_internal_id_to_idx': drug_internal_id_to_idx,
    'effect_id_to_idx': effect_id_to_idx,
    
    # Edges: Drug → Effect
    'drug_effect_edge_index': drug_effect_edge_index,
    'drug_effect_edge_attr': drug_effect_edge_attr,
    
    # Metadata
    'num_drugs': len(drug_pyg_objects),
    'num_effects': len(effect_pyg_objects),
    'num_proteins': len(protein_pyg_objects),
    'num_drug_effect_edges': drug_effect_edge_index.shape[1]
}

print("\n✅ Graph data structure created:")
print(f"  Drugs:           {graph_data['num_drugs']:,}")
print(f"  Effects:         {graph_data['num_effects']:,}")
print(f"  Proteins:        {graph_data['num_proteins']:,}")
print(f"  Drug→Effect:     {graph_data['num_drug_effect_edges']:,}")



CREATING COMPREHENSIVE NODE & EDGE MAPPINGS

✅ Graph data structure created:
  Drugs:           3,127
  Effects:         1,065
  Proteins:        2,053
  Drug→Effect:     5,633


In [14]:
import torch
import numpy as np
import pandas as pd
from torch_geometric.data import HeteroData

print("="*80)
print("STEP 1: BUILD NODE MAPPINGS")
print("="*80)

drug_ids = drug_nodes['drug_internal_id'].values
protein_ids = protein_nodes['protein_id'].unique()
effect_ids = drug_indications['effect_id'].unique()

drug_to_idx = {did: i for i, did in enumerate(drug_ids)}
protein_to_idx = {pid: i for i, pid in enumerate(protein_ids)}
effect_to_idx = {eid: i for i, eid in enumerate(effect_ids)}

num_drugs = len(drug_to_idx)
num_proteins = len(protein_to_idx)
num_effects = len(effect_to_idx)

print(f"  Drugs:    {num_drugs:,}")
print(f"  Proteins: {num_proteins:,}")
print(f"  Effects:  {num_effects:,}")

STEP 1: BUILD NODE MAPPINGS
  Drugs:    3,127
  Proteins: 2,053
  Effects:  1,065


In [15]:
print("\n" + "="*80)
print("STEP 1.5: FILTER TO CONNECTED NODES ONLY")
print("="*80)

# ── Collect all nodes that appear in at least ONE edge across ALL splits ──
all_dp_nodes_src = set()
all_dp_nodes_tgt = set()
all_di_nodes_src = set()
all_di_nodes_tgt = set()

for _, row in drugs_interactions_train.iterrows():
    all_dp_nodes_src.add(int(row['drug_internal_id']))
    all_dp_nodes_tgt.add(str(row['protein_id']))

for _, row in drugs_interactions_validation.iterrows():
    all_dp_nodes_src.add(int(row['drug_internal_id']))
    all_dp_nodes_tgt.add(str(row['protein_id']))

for _, row in drugs_interactions_test.iterrows():
    all_dp_nodes_src.add(int(row['drug_internal_id']))
    all_dp_nodes_tgt.add(str(row['protein_id']))

for _, row in drug_indications.iterrows():
    all_di_nodes_src.add(int(row['drug_internal_id']))
    all_di_nodes_tgt.add(str(row['effect_id']))

# Union of all connected nodes (no feature-availability filtering)
connected_drugs = all_dp_nodes_src | all_di_nodes_src
connected_proteins = all_dp_nodes_tgt
connected_effects = all_di_nodes_tgt

print(f"  Connected nodes found in edges:")
print(f"    Drugs:    {len(connected_drugs):,}")
print(f"    Proteins: {len(connected_proteins):,}")
print(f"    Effects:  {len(connected_effects):,}")

# ── Filter and reindex nodes ──
drug_ids_filtered = np.array(sorted(list(connected_drugs)))
protein_ids_filtered = np.array(sorted(list(connected_proteins)))
effect_ids_filtered = np.array(sorted(list(connected_effects)))

# Create NEW mappings with only connected nodes
drug_to_idx = {did: i for i, did in enumerate(drug_ids_filtered)}
protein_to_idx = {pid: i for i, pid in enumerate(protein_ids_filtered)}
effect_to_idx = {eid: i for i, eid in enumerate(effect_ids_filtered)}

num_drugs = len(drug_to_idx)
num_proteins = len(protein_to_idx)
num_effects = len(effect_to_idx)

total_nodes = num_drugs + num_proteins + num_effects
print(f"\n  Final node counts:")
print(f"    Drugs:    {num_drugs:,}")
print(f"    Proteins: {num_proteins:,}")
print(f"    Effects:  {num_effects:,}")
print(f"    Total:    {total_nodes:,}")

print("\n" + "="*80)
print("✓ FILTERED TO CONNECTED NODES ONLY (identity features — no SMILES/ESM2 needed)")
print("="*80)


STEP 1.5: FILTER TO CONNECTED NODES ONLY
  Connected nodes found in edges:
    Drugs:    3,071
    Proteins: 1,966
    Effects:  1,065

  Final node counts:
    Drugs:    3,071
    Proteins: 1,966
    Effects:  1,065
    Total:    6,102

✓ FILTERED TO CONNECTED NODES ONLY (identity features — no SMILES/ESM2 needed)


In [16]:
print("\n" + "="*80)
print("STEP 2: PROCESS DRUG-PROTEIN EDGES (TIME-AWARE SPLIT)")
print("="*80)

def edges_to_tensor(df, src_col, tgt_col, src_map, tgt_map):
    """Convert edges to tensor, filtering out missing nodes."""
    src_indices = []
    tgt_indices = []
    for _, row in df.iterrows():
        src_id = int(row[src_col])
        tgt_id = str(row[tgt_col])
        if src_id in src_map and tgt_id in tgt_map:
            src_indices.append(src_map[src_id])
            tgt_indices.append(tgt_map[tgt_id])
    return torch.LongTensor([src_indices, tgt_indices]) if src_indices else torch.zeros((2, 0), dtype=torch.long)

# Drug-Protein edges from pre-split data
dp_train_edge_index = edges_to_tensor(drugs_interactions_train, 'drug_internal_id', 'protein_id', drug_to_idx, protein_to_idx)
dp_val_edge_index = edges_to_tensor(drugs_interactions_validation, 'drug_internal_id', 'protein_id', drug_to_idx, protein_to_idx)
dp_test_edge_index = edges_to_tensor(drugs_interactions_test, 'drug_internal_id', 'protein_id', drug_to_idx, protein_to_idx)

print(f"  Drug-Protein Training edges:   {dp_train_edge_index.shape[1]:,}")
print(f"  Drug-Protein Validation edges: {dp_val_edge_index.shape[1]:,}")
print(f"  Drug-Protein Test edges:       {dp_test_edge_index.shape[1]:,}")


STEP 2: PROCESS DRUG-PROTEIN EDGES (TIME-AWARE SPLIT)
  Drug-Protein Training edges:   10,409
  Drug-Protein Validation edges: 1,901
  Drug-Protein Test edges:       1,901


In [17]:
print("\n" + "="*80)
print("STEP 3: PROCESS DRUG-INDICATION EDGES (RANDOM SPLIT)")
print("="*80)

# Create edges from drug_indications
di_edges = edges_to_tensor(drug_indications, 'drug_internal_id', 'effect_id', drug_to_idx, effect_to_idx)
print(f"  Total drug-indication edges: {di_edges.shape[1]:,}")

# Random 80/10/10 split (seed for reproducibility)
np.random.seed(42)
num_di_edges = di_edges.shape[1]
perm = np.random.permutation(num_di_edges)
train_size = int(0.8 * num_di_edges)
val_size = int(0.1 * num_di_edges)

di_train_idx = perm[:train_size]
di_val_idx = perm[train_size:train_size + val_size]
di_test_idx = perm[train_size + val_size:]

di_train_edge_index = di_edges[:, di_train_idx]
di_val_edge_index = di_edges[:, di_val_idx]
di_test_edge_index = di_edges[:, di_test_idx]

print(f"  Drug-Indication Training edges:   {di_train_edge_index.shape[1]:,}")
print(f"  Drug-Indication Validation edges: {di_val_edge_index.shape[1]:,}")
print(f"  Drug-Indication Test edges:       {di_test_edge_index.shape[1]:,}")


STEP 3: PROCESS DRUG-INDICATION EDGES (RANDOM SPLIT)
  Total drug-indication edges: 7,086
  Drug-Indication Training edges:   5,668
  Drug-Indication Validation edges: 708
  Drug-Indication Test edges:       710


In [18]:
print("\n" + "="*80)
print("STEP 4: SETUP DYNAMIC NEGATIVE SAMPLING - DRUG-PROTEIN")
print("="*80)

# Create mapping from ChEMBL drug_id to drug_internal_id
chembl_to_internal = dict(zip(drug_nodes['drug_id'], drug_nodes['drug_internal_id']))

def get_existing_edges_set(edge_index):
    """Convert edge_index tensor to set of (src, tgt) tuples."""
    if edge_index.shape[1] == 0:
        return set()
    return set(map(tuple, edge_index.t().numpy()))

# Get nodes that have at least one positive edge in training
train_drugs_with_pos = set(dp_train_edge_index[0].numpy())
train_proteins_with_pos = set(dp_train_edge_index[1].numpy())

print(f"  Drugs with positive edges in train:    {len(train_drugs_with_pos):,}")
print(f"  Proteins with positive edges in train: {len(train_proteins_with_pos):,}")

# Preprocess verified negatives (convert to indices once)
verified_dp_train = []
for _, row in verified_negatives_protien_interactions_train.iterrows():
    drug_chembl = str(row['drug_id'])
    protein_id = str(row['protein_id'])
    if drug_chembl in chembl_to_internal and protein_id in protein_to_idx:
        drug_internal_id = chembl_to_internal[drug_chembl]
        if drug_internal_id in drug_to_idx:
            drug_idx = drug_to_idx[drug_internal_id]
            protein_idx = protein_to_idx[protein_id]
            if drug_idx in train_drugs_with_pos and protein_idx in train_proteins_with_pos:
                verified_dp_train.append((drug_idx, protein_idx))

verified_dp_test = []
for _, row in verified_negatives_protien_interactions_test.iterrows():
    drug_chembl = str(row['drug_id'])
    protein_id = str(row['protein_id'])
    if drug_chembl in chembl_to_internal and protein_id in protein_to_idx:
        drug_internal_id = chembl_to_internal[drug_chembl]
        if drug_internal_id in drug_to_idx:
            drug_idx = drug_to_idx[drug_internal_id]
            protein_idx = protein_to_idx[protein_id]
            if drug_idx in train_drugs_with_pos and protein_idx in train_proteins_with_pos:
                verified_dp_test.append((drug_idx, protein_idx))

verified_dp_train = list(set(verified_dp_train))
verified_dp_test = list(set(verified_dp_test))

print(f"  Verified DP negatives (train): {len(verified_dp_train):,}")
print(f"  Verified DP negatives (test):  {len(verified_dp_test):,}")

def sample_negatives_dp_dynamic(num_samples, verified_negs, existing_edges, valid_srcs, valid_tgts, ratio=0.5):
    """
    Dynamic drug-protein negative sampling.
    50% verified + 50% random (from valid nodes).
    Called during each training step - ensures fresh negatives each time.
    """
    verified_count = min(len(verified_negs), int(num_samples * ratio))
    random_count = num_samples - verified_count
    
    negatives = []
    
    # Add verified negatives
    if verified_negs:
        for edge_idx in np.random.choice(len(verified_negs), min(verified_count, len(verified_negs)), replace=False):
            negatives.append(verified_negs[edge_idx])
    
    # Add random negatives from valid nodes
    attempts = 0
    max_attempts = random_count * 20
    while len(negatives) < num_samples and attempts < max_attempts:
        drug_idx = np.random.choice(list(valid_srcs))
        protein_idx = np.random.choice(list(valid_tgts))
        pair = (drug_idx, protein_idx)
        if pair not in existing_edges and pair not in negatives:
            negatives.append(pair)
        attempts += 1
    
    # Fill remaining
    while len(negatives) < num_samples:
        drug_idx = np.random.choice(list(valid_srcs))
        protein_idx = np.random.choice(list(valid_tgts))
        pair = (drug_idx, protein_idx)
        if pair not in existing_edges and pair not in negatives:
            negatives.append(pair)
    
    return negatives[:num_samples]

# Pre-compute all positive edges for fast lookup during training
all_pos_dp_edges = torch.cat([dp_train_edge_index, dp_val_edge_index, dp_test_edge_index], dim=1)
existing_dp = get_existing_edges_set(all_pos_dp_edges)

print(f"  Total positive DP edges (all splits): {len(existing_dp):,}")


STEP 4: SETUP DYNAMIC NEGATIVE SAMPLING - DRUG-PROTEIN
  Drugs with positive edges in train:    1,402
  Proteins with positive edges in train: 1,966
  Verified DP negatives (train): 43,867
  Verified DP negatives (test):  40,456
  Total positive DP edges (all splits): 14,211


In [19]:
print("\n" + "="*80)
print("STEP 5: SETUP DYNAMIC NEGATIVE SAMPLING - DRUG-INDICATION")
print("="*80)

# Create mapping from effect_name to effect_id
effect_name_to_id = dict(zip(drug_indications['effect_name'], drug_indications['effect_id']))

# Get nodes that have at least one positive edge in training
train_drugs_di = set(di_train_edge_index[0].numpy())
train_effects_di = set(di_train_edge_index[1].numpy())

print(f"  Drugs with positive edges in train:    {len(train_drugs_di):,}")
print(f"  Effects with positive edges in train:  {len(train_effects_di):,}")

# Preprocess hard negatives
hard_neg_edges = []
for _, row in failed_indication_hard_negatives.iterrows():
    drug_chembl = str(row['drug_id'])
    effect_name = str(row['effect_name'])
    if drug_chembl in chembl_to_internal and effect_name in effect_name_to_id:
        drug_internal_id = chembl_to_internal[drug_chembl]
        effect_id = effect_name_to_id[effect_name]
        if drug_internal_id in drug_to_idx and effect_id in effect_to_idx:
            drug_idx = drug_to_idx[drug_internal_id]
            effect_idx = effect_to_idx[effect_id]
            if drug_idx in train_drugs_di and effect_idx in train_effects_di:
                hard_neg_edges.append((drug_idx, effect_idx))

# Preprocess medium negatives
med_neg_edges = []
for _, row in failed_indication_medium_negatives.iterrows():
    drug_chembl = str(row['drug_id'])
    effect_name = str(row['effect_name'])
    if drug_chembl in chembl_to_internal and effect_name in effect_name_to_id:
        drug_internal_id = chembl_to_internal[drug_chembl]
        effect_id = effect_name_to_id[effect_name]
        if drug_internal_id in drug_to_idx and effect_id in effect_to_idx:
            drug_idx = drug_to_idx[drug_internal_id]
            effect_idx = effect_to_idx[effect_id]
            if drug_idx in train_drugs_di and effect_idx in train_effects_di:
                med_neg_edges.append((drug_idx, effect_idx))

hard_neg_edges = list(set(hard_neg_edges))
med_neg_edges = list(set(med_neg_edges))

print(f"  Hard negatives (Phase 3 fails):        {len(hard_neg_edges):,}")
print(f"  Medium negatives (Phase 2 or less):    {len(med_neg_edges):,}")

def sample_negatives_di_dynamic(num_samples, hard_negs, med_negs, existing_edges, valid_srcs, valid_tgts):
    """
    Dynamic drug-indication negative sampling.
    33% hard + 33% medium + 33% random (from valid nodes).
    Called during each training step - ensures fresh negatives each time.
    """
    hard_count = min(len(hard_negs), num_samples // 3)
    med_count = min(len(med_negs), num_samples // 3)
    random_count = num_samples - hard_count - med_count
    
    negatives = []
    
    # Add hard negatives
    if hard_negs:
        for idx in np.random.choice(len(hard_negs), min(hard_count, len(hard_negs)), replace=False):
            negatives.append(hard_negs[idx])
    
    # Add medium negatives
    if med_negs:
        for idx in np.random.choice(len(med_negs), min(med_count, len(med_negs)), replace=False):
            negatives.append(med_negs[idx])
    
    # Add random negatives from valid nodes
    all_neg_set = set(hard_negs + med_negs)
    attempts = 0
    max_attempts = random_count * 20
    
    while len(negatives) < num_samples and attempts < max_attempts:
        drug_idx = np.random.choice(list(valid_srcs))
        effect_idx = np.random.choice(list(valid_tgts))
        pair = (drug_idx, effect_idx)
        if pair not in existing_edges and pair not in all_neg_set and pair not in negatives:
            negatives.append(pair)
        attempts += 1
    
    # Fill remaining
    while len(negatives) < num_samples:
        drug_idx = np.random.choice(list(valid_srcs))
        effect_idx = np.random.choice(list(valid_tgts))
        pair = (drug_idx, effect_idx)
        if pair not in existing_edges and pair not in negatives:
            negatives.append(pair)
    
    return negatives[:num_samples]

# Pre-compute all positive edges for fast lookup during training
all_pos_di_edges = torch.cat([di_train_edge_index, di_val_edge_index, di_test_edge_index], dim=1)
existing_di = get_existing_edges_set(all_pos_di_edges)

print(f"  Total positive DI edges (all splits): {len(existing_di):,}")


STEP 5: SETUP DYNAMIC NEGATIVE SAMPLING - DRUG-INDICATION
  Drugs with positive edges in train:    2,474
  Effects with positive edges in train:  971
  Hard negatives (Phase 3 fails):        4,911
  Medium negatives (Phase 2 or less):    8,783
  Total positive DI edges (all splits): 7,086


In [20]:
print("\n" + "="*80)
print("STEP 6: BUILD FEATURE TENSORS & EDGE INDEX DICTS FOR HGNN")
print("="*80)

# ── Drug PyG objects indexed by drug_to_idx ───────────────────────────────────
# drug_ids_filtered[i]  →  drug_to_idx[drug_ids_filtered[i]] = i
# We need drug_pyg_indexed[i] = the PyG Data object for drug_ids_filtered[i]
# so that edge indices in dp_train_edge_index (which use drug_to_idx) directly
# index into drug_pyg_indexed.

drug_internal_id_to_pyg = {d.drug_internal_id: d for d in drug_pyg_objects}

drug_pyg_indexed = []
_missing_drug = 0
for did in drug_ids_filtered:
    if did in drug_internal_id_to_pyg:
        drug_pyg_indexed.append(drug_internal_id_to_pyg[did])
    else:
        _missing_drug += 1
        # Dummy single-atom graph (zero features) for drugs with no SMILES
        dummy = Data(
            x=torch.zeros(1, total_atom_dim),
            edge_index=torch.zeros(2, 0, dtype=torch.long),
            edge_attr=torch.zeros(0, total_bond_dim),
            drug_internal_id=int(did), drug_id='UNKNOWN', drug_name='Unknown'
        )
        drug_pyg_indexed.append(dummy)

print(f"  Drug PyG objects (indexed to drug_to_idx):  {len(drug_pyg_indexed)}"
      f"  ({_missing_drug} dummy for missing SMILES)")

# ── Protein feature tensor indexed by protein_to_idx ─────────────────────────
# protein_ids_filtered[i]  →  protein_to_idx[protein_ids_filtered[i]] = i
# protein_features_tensor[i]  =  ESM-2 embedding for that protein

protein_id_to_pyg = {p.protein_id: p for p in protein_pyg_objects}
_sample_prot_dim = protein_pyg_objects[0].x.squeeze().shape[0]

protein_feats = []
for pid in protein_ids_filtered:
    if pid in protein_id_to_pyg:
        protein_feats.append(protein_id_to_pyg[pid].x.squeeze())
    else:
        protein_feats.append(torch.zeros(_sample_prot_dim))

protein_features_tensor = torch.stack(protein_feats)
print(f"  Protein feature tensor (ESM-2):             {protein_features_tensor.shape}")

# ── Effect feature tensor indexed by effect_to_idx ───────────────────────────
# effect_ids_filtered[i]  →  effect_to_idx[effect_ids_filtered[i]] = i
# effect_features_tensor[i]  =  learned embedding for that effect

effect_id_to_pyg = {e.effect_id: e for e in effect_pyg_objects}
_sample_eff_dim = effect_pyg_objects[0].x.squeeze().shape[0]

effect_feats = []
for eid in effect_ids_filtered:
    if eid in effect_id_to_pyg:
        effect_feats.append(effect_id_to_pyg[eid].x.squeeze())
    else:
        effect_feats.append(torch.zeros(_sample_eff_dim))

effect_features_tensor = torch.stack(effect_feats)
print(f"  Effect feature tensor (learned):            {effect_features_tensor.shape}")

# ── Edge index dicts for each split (cumulative for message passing) ──────────
# Training graph  : train edges only
# Validation graph: train + val edges  (evaluate on val edges)
# Test graph      : train + val + test (evaluate on test edges)

dp_val_cumulative  = torch.cat([dp_train_edge_index, dp_val_edge_index], dim=1)
di_val_cumulative  = torch.cat([di_train_edge_index, di_val_edge_index], dim=1)

dp_test_cumulative = torch.cat([dp_train_edge_index, dp_val_edge_index, dp_test_edge_index], dim=1)
di_test_cumulative = torch.cat([di_train_edge_index, di_val_edge_index, di_test_edge_index], dim=1)

def build_edge_index_dict(dp_edges, di_edges):
    """Build heterogeneous edge_index_dict including reverse edges for bidirectional MP."""
    return {
        ('drug',    'binds_to',     'protein'): dp_edges,
        ('protein', 'rev_binds_to', 'drug'):    dp_edges.flip(0),
        ('drug',    'treats',       'effect'):  di_edges,
        ('effect',  'rev_treats',   'drug'):    di_edges.flip(0),
    }

train_edge_index_dict = build_edge_index_dict(dp_train_edge_index, di_train_edge_index)
val_edge_index_dict   = build_edge_index_dict(dp_val_cumulative,   di_val_cumulative)
test_edge_index_dict  = build_edge_index_dict(dp_test_cumulative,  di_test_cumulative)

print(f"\n  Train edge dict:  DP={dp_train_edge_index.shape[1]:,}  DI={di_train_edge_index.shape[1]:,}")
print(f"  Val edge dict:    DP={dp_val_cumulative.shape[1]:,}  DI={di_val_cumulative.shape[1]:,}")
print(f"  Test edge dict:   DP={dp_test_cumulative.shape[1]:,}  DI={di_test_cumulative.shape[1]:,}")

# Sanity check: edge indices must be within node counts
assert dp_train_edge_index[0].max() < num_drugs,    "DP train: drug idx out of bounds"
assert dp_train_edge_index[1].max() < num_proteins, "DP train: protein idx out of bounds"
assert di_train_edge_index[0].max() < num_drugs,    "DI train: drug idx out of bounds"
assert di_train_edge_index[1].max() < num_effects,  "DI train: effect idx out of bounds"
print("\n  ✓ All edge indices within node bounds")

print("\n" + "="*80)
print("✓ STEP 6 COMPLETE — feature tensors and edge index dicts ready for HGNN")
print("="*80)



STEP 6: BUILD FEATURE TENSORS & EDGE INDEX DICTS FOR HGNN
  Drug PyG objects (indexed to drug_to_idx):  3071  (647 dummy for missing SMILES)
  Protein feature tensor (ESM-2):             torch.Size([1966, 2560])
  Effect feature tensor (learned):            torch.Size([1065, 32])

  Train edge dict:  DP=10,409  DI=5,668
  Val edge dict:    DP=12,310  DI=6,376
  Test edge dict:   DP=14,211  DI=7,086

  ✓ All edge indices within node bounds

✓ STEP 6 COMPLETE — feature tensors and edge index dicts ready for HGNN


In [21]:
print("\n" + "="*80)
print("STEP 7: SAVE NODE MAPPINGS, FEATURE TENSORS & SAMPLING CONFIG")
print("="*80)

mappings = {
    'drug_to_idx':    drug_to_idx,
    'protein_to_idx': protein_to_idx,
    'effect_to_idx':  effect_to_idx,
}

edge_dicts = {
    'train': train_edge_index_dict,
    'val':   val_edge_index_dict,
    'test':  test_edge_index_dict,
    # individual split edge indices for evaluation
    'dp_train': dp_train_edge_index,
    'dp_val':   dp_val_edge_index,
    'dp_test':  dp_test_edge_index,
    'di_train': di_train_edge_index,
    'di_val':   di_val_edge_index,
    'di_test':  di_test_edge_index,
}

sampling_config = {
    'verified_dp_train':      verified_dp_train,
    'verified_dp_test':       verified_dp_test,
    'hard_neg_edges':         hard_neg_edges,
    'med_neg_edges':          med_neg_edges,
    'existing_dp':            existing_dp,
    'existing_di':            existing_di,
    'train_drugs_with_pos':   train_drugs_with_pos,
    'train_proteins_with_pos': train_proteins_with_pos,
    'train_drugs_di':         train_drugs_di,
    'train_effects_di':       train_effects_di,
}

torch.save(mappings,        'hetero_node_mappings.pt')
torch.save(edge_dicts,      'hgnn_edge_dicts.pt')
torch.save(sampling_config, 'dynamic_sampling_config.pt')
torch.save(protein_features_tensor, 'protein_features_tensor.pt')
torch.save(effect_features_tensor,  'effect_features_tensor.pt')

print("✓ Saved hetero_node_mappings.pt")
print("✓ Saved hgnn_edge_dicts.pt")
print("✓ Saved dynamic_sampling_config.pt")
print("✓ Saved protein_features_tensor.pt")
print("✓ Saved effect_features_tensor.pt")

print("\n" + "="*80)
print("✅ READY FOR HGNN TRAINING WITH DYNAMIC NEGATIVE SAMPLING!")
print("="*80)
print(f"\n  Drugs:    {num_drugs:,}   |   Proteins: {num_proteins:,}   |   Effects: {num_effects:,}")
print(f"  Drug features:    molecular graphs (GAT encoder, {total_atom_dim}-dim atoms, {total_bond_dim}-dim bonds)")
print(f"  Protein features: ESM-2 embeddings ({protein_features_tensor.shape[1]}-dim)")
print(f"  Effect features:  learned embeddings ({effect_features_tensor.shape[1]}-dim)")



STEP 7: SAVE NODE MAPPINGS, FEATURE TENSORS & SAMPLING CONFIG
✓ Saved hetero_node_mappings.pt
✓ Saved hgnn_edge_dicts.pt
✓ Saved dynamic_sampling_config.pt
✓ Saved protein_features_tensor.pt
✓ Saved effect_features_tensor.pt

✅ READY FOR HGNN TRAINING WITH DYNAMIC NEGATIVE SAMPLING!

  Drugs:    3,071   |   Proteins: 1,966   |   Effects: 1,065
  Drug features:    molecular graphs (GAT encoder, 57-dim atoms, 4-dim bonds)
  Protein features: ESM-2 embeddings (2560-dim)
  Effect features:  learned embeddings (32-dim)


In [22]:
print("\n" + "="*80)
print("STEP 7b: GRAPH SUMMARY")
print("="*80)

print(f"\nNode counts (connected nodes only):")
print(f"  Drugs:    {num_drugs:,}")
print(f"  Proteins: {num_proteins:,}")
print(f"  Effects:  {num_effects:,}")
print(f"  Total:    {num_drugs + num_proteins + num_effects:,}")

print(f"\nEdge splits (Drug-Protein):")
print(f"  Train: {dp_train_edge_index.shape[1]:,}")
print(f"  Val:   {dp_val_edge_index.shape[1]:,}")
print(f"  Test:  {dp_test_edge_index.shape[1]:,}")

print(f"\nEdge splits (Drug-Indication):")
print(f"  Train: {di_train_edge_index.shape[1]:,}")
print(f"  Val:   {di_val_edge_index.shape[1]:,}")
print(f"  Test:  {di_test_edge_index.shape[1]:,}")

print(f"\nNode features:")
print(f"  Drugs:    molecular graph (atoms={total_atom_dim}-dim one-hot, bonds={total_bond_dim}-dim one-hot) → GAT encoder")
print(f"  Proteins: pre-computed ESM-2  {protein_features_tensor.shape}")
print(f"  Effects:  learned embeddings  {effect_features_tensor.shape}")

print(f"\nNegative sampling pool:")
print(f"  DP verified negatives (train): {len(verified_dp_train):,}")
print(f"  DI hard negatives:             {len(hard_neg_edges):,}")
print(f"  DI medium negatives:           {len(med_neg_edges):,}")

print("\n" + "="*80)
print("✅ ALL VARIABLES READY — proceed to model definition and training")
print("="*80)



STEP 7b: GRAPH SUMMARY

Node counts (connected nodes only):
  Drugs:    3,071
  Proteins: 1,966
  Effects:  1,065
  Total:    6,102

Edge splits (Drug-Protein):
  Train: 10,409
  Val:   1,901
  Test:  1,901

Edge splits (Drug-Indication):
  Train: 5,668
  Val:   708
  Test:  710

Node features:
  Drugs:    molecular graph (atoms=57-dim one-hot, bonds=4-dim one-hot) → GAT encoder
  Proteins: pre-computed ESM-2  torch.Size([1966, 2560])
  Effects:  learned embeddings  torch.Size([1065, 32])

Negative sampling pool:
  DP verified negatives (train): 43,867
  DI hard negatives:             4,911
  DI medium negatives:           8,783

✅ ALL VARIABLES READY — proceed to model definition and training


In [23]:
print("\n" + "="*80)
print("EXAMPLE: HOW TO USE DYNAMIC NEGATIVE SAMPLING IN TRAINING")
print("="*80)

# Example usage during training loop:
print("\nDuring training, sample negatives dynamically like this:\n")

print("# For each batch of drug-protein edges:")
sample_neg_dp = sample_negatives_dp_dynamic(
    num_samples=100,
    verified_negs=verified_dp_train,
    existing_edges=existing_dp,
    valid_srcs=train_drugs_with_pos,
    valid_tgts=train_proteins_with_pos
)
print(f"  Generated {len(sample_neg_dp)} fresh negatives for DP")

print("\n# For each batch of drug-indication edges:")
sample_neg_di = sample_negatives_di_dynamic(
    num_samples=100,
    hard_negs=hard_neg_edges,
    med_negs=med_neg_edges,
    existing_edges=existing_di,
    valid_srcs=train_drugs_di,
    valid_tgts=train_effects_di
)
print(f"  Generated {len(sample_neg_di)} fresh negatives for DI")
print("    - This mix includes 33% hard + 33% medium + 33% random")

print("\n✅ These functions will be called EVERY TRAINING STEP/EPOCH")
print("   ensuring the model never memorizes specific negative samples!")

print("\n" + "="*80)
print("KEY VARIABLES FOR MODEL TRAINING:")
print("="*80)
print(f"✓ hetero_train, hetero_val, hetero_test (HeteroData objects)")
print(f"✓ sample_negatives_dp_dynamic() - Call during DP training")
print(f"✓ sample_negatives_di_dynamic() - Call during DI training")
print(f"✓ verified_dp_train, verified_dp_test - Verified negative lists")
print(f"✓ hard_neg_edges, med_neg_edges - Pre-processed failed indications")
print(f"✓ existing_dp, existing_di - All positive edges (for validation)")
print(f"✓ Valid node sets: train_drugs_with_pos, train_proteins_with_pos, etc.")


EXAMPLE: HOW TO USE DYNAMIC NEGATIVE SAMPLING IN TRAINING

During training, sample negatives dynamically like this:

# For each batch of drug-protein edges:
  Generated 100 fresh negatives for DP

# For each batch of drug-indication edges:
  Generated 100 fresh negatives for DI
    - This mix includes 33% hard + 33% medium + 33% random

✅ These functions will be called EVERY TRAINING STEP/EPOCH
   ensuring the model never memorizes specific negative samples!

KEY VARIABLES FOR MODEL TRAINING:
✓ hetero_train, hetero_val, hetero_test (HeteroData objects)
✓ sample_negatives_dp_dynamic() - Call during DP training
✓ sample_negatives_di_dynamic() - Call during DI training
✓ verified_dp_train, verified_dp_test - Verified negative lists
✓ hard_neg_edges, med_neg_edges - Pre-processed failed indications
✓ existing_dp, existing_di - All positive edges (for validation)
✓ Valid node sets: train_drugs_with_pos, train_proteins_with_pos, etc.


In [24]:
import time
from sklearn.metrics import roc_auc_score, average_precision_score

print("="*80)
print("EVALUATION METRICS (Reviewer-Requested)")
print("="*80)

def print_metrics(metrics, prefix=""):
    """Pretty-print a metrics dict."""
    print(f"  {prefix}ROC-AUC: {metrics['roc_auc']:.4f}   PR-AUC: {metrics['pr_auc']:.4f}")
    print(f"  {prefix}MRR:     {metrics['mrr']:.4f}   Hits@1: {metrics['hits@1']:.4f}  "
          f"Hits@3: {metrics['hits@3']:.4f}  Hits@10: {metrics['hits@10']:.4f}")

def get_peak_vram_mb():
    """Return peak GPU VRAM allocated in MB (0 if CPU-only)."""
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / 1024**2
    return 0.0

def reset_vram_tracker():
    """Reset the peak VRAM counter before each epoch."""
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

print("  ✓ Evaluation utilities ready")
print("  Protocol: Standard filtered ranking (Bordes et al., 2013)")
print("    - Each positive edge ranked against ALL entities of the target type")
print("    - Other known positives filtered out (score → -inf)")
print("    - MRR, Hits@k computed per-edge then averaged")
print("    - ROC-AUC, PR-AUC derived from the full ranking scores")

EVALUATION METRICS (Reviewer-Requested)
  ✓ Evaluation utilities ready
  Protocol: Standard filtered ranking (Bordes et al., 2013)
    - Each positive edge ranked against ALL entities of the target type
    - Other known positives filtered out (score → -inf)
    - MRR, Hits@k computed per-edge then averaged
    - ROC-AUC, PR-AUC derived from the full ranking scores


In [25]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, SAGEConv, global_mean_pool
from torch_geometric.data import Batch
import os

os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("="*80)
print("SCALING ABLATION STUDY  —  PharmacologyHeteroGNN")
print("Track 1: Data Scaling    (shared_dim=256, vary training data fraction)")
print("Track 2: Parameter Scaling (100% data, vary shared_dim)")
print("="*80)

# ─────────────────────────────────────────────────────────────────────────────
# 1.  Drug Molecular Encoder  (unchanged from full model)
# ─────────────────────────────────────────────────────────────────────────────

class DrugMolecularEncoder(nn.Module):
    def __init__(self, node_feat_dim, edge_feat_dim, hidden_dim, output_dim,
                 num_layers=3, heads=4, dropout=0.1):
        super().__init__()
        self.node_enc = nn.Linear(node_feat_dim, hidden_dim)
        self.edge_enc = nn.Linear(edge_feat_dim, hidden_dim)
        self.gat_layers = nn.ModuleList()
        for i in range(num_layers):
            in_dim = hidden_dim if i == 0 else hidden_dim * heads
            self.gat_layers.append(
                GATConv(in_dim, hidden_dim, heads=heads, edge_dim=hidden_dim, dropout=dropout)
            )
        self.out_proj = nn.Linear(hidden_dim * heads, output_dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, batch_data):
        x         = self.node_enc(batch_data.x)
        edge_attr = self.edge_enc(batch_data.edge_attr)
        for gat in self.gat_layers:
            x = gat(x, batch_data.edge_index, edge_attr=edge_attr)
            x = F.elu(x)
            x = self.drop(x)
        return self.out_proj(global_mean_pool(x, batch_data.batch))


# ─────────────────────────────────────────────────────────────────────────────
# 2.  Heterogeneous GraphSAGE Layer  (unchanged from full model)
# ─────────────────────────────────────────────────────────────────────────────

class HeteroSAGELayer(nn.Module):
    def __init__(self, hidden_dim, edge_types):
        super().__init__()
        self.convs = nn.ModuleDict()
        dst_types  = set()
        for src, rel, dst in edge_types:
            self.convs[f"{src}__{rel}__{dst}"] = SAGEConv(
                (hidden_dim, hidden_dim), hidden_dim, aggr='mean'
            )
            dst_types.add(dst)
        self.norms = nn.ModuleDict({t: nn.LayerNorm(hidden_dim) for t in dst_types})

    def forward(self, x_dict, edge_index_dict):
        contributions = {ntype: [] for ntype in x_dict}
        for (src_type, rel, dst_type), ei in edge_index_dict.items():
            key = f"{src_type}__{rel}__{dst_type}"
            if key in self.convs and ei.shape[1] > 0:
                out = self.convs[key]((x_dict[src_type], x_dict[dst_type]), ei)
                contributions[dst_type].append(out)
        out_dict = {}
        for ntype, feats in x_dict.items():
            if contributions[ntype]:
                agg = torch.stack(contributions[ntype]).mean(dim=0)
                out_dict[ntype] = self.norms[ntype](agg + feats)
            else:
                out_dict[ntype] = feats
        return out_dict


# ─────────────────────────────────────────────────────────────────────────────
# 3.  PharmacologyHeteroGNN  (shared_dim is a free parameter for Track 2)
# ─────────────────────────────────────────────────────────────────────────────

class PharmacologyHeteroGNN(nn.Module):
    _EDGE_TYPES = [
        ('drug',    'binds_to',     'protein'),
        ('protein', 'rev_binds_to', 'drug'),
        ('drug',    'treats',       'effect'),
        ('effect',  'rev_treats',   'drug'),
    ]

    def __init__(self, cfg):
        super().__init__()
        sd = cfg['shared_dim']
        self.drug_encoder = DrugMolecularEncoder(
            node_feat_dim = cfg['drug_node_feat_dim'],
            edge_feat_dim = cfg['drug_edge_feat_dim'],
            hidden_dim    = cfg['drug_hidden_dim'],
            output_dim    = sd,
            num_layers    = cfg['drug_num_layers'],
            heads         = cfg['drug_heads'],
        )
        self.protein_proj = nn.Sequential(
            nn.Linear(cfg['protein_feat_dim'], sd), nn.LayerNorm(sd), nn.ReLU(), nn.Dropout(0.1)
        )
        self.effect_proj = nn.Sequential(
            nn.Linear(cfg['effect_feat_dim'], sd), nn.LayerNorm(sd), nn.ReLU(), nn.Dropout(0.1)
        )
        self.hetero_layers = nn.ModuleList([
            HeteroSAGELayer(sd, self._EDGE_TYPES)
            for _ in range(cfg['num_hetero_layers'])
        ])
        self.dp_head = nn.Sequential(
            nn.Linear(sd * 2, sd), nn.ReLU(), nn.Dropout(0.2), nn.Linear(sd, 1)
        )
        self.di_head = nn.Sequential(
            nn.Linear(sd * 2, sd), nn.ReLU(), nn.Dropout(0.2), nn.Linear(sd, 1)
        )

    def encode(self, drug_batch, protein_features, effect_features, edge_index_dict):
        x_dict = {
            'drug':    self.drug_encoder(drug_batch),
            'protein': self.protein_proj(protein_features),
            'effect':  self.effect_proj(effect_features),
        }
        for layer in self.hetero_layers:
            x_dict = layer(x_dict, edge_index_dict)
            for ntype in x_dict:
                x_dict[ntype] = F.relu(x_dict[ntype])
        return x_dict

    def score_dp(self, drug_emb, protein_emb):
        return self.dp_head(torch.cat([drug_emb, protein_emb], dim=-1)).squeeze(-1)

    def score_di(self, drug_emb, effect_emb):
        return self.di_head(torch.cat([drug_emb, effect_emb], dim=-1)).squeeze(-1)

    @staticmethod
    def margin_loss(pos_score, neg_score, margin=1.0):
        return F.relu(margin - pos_score + neg_score).mean()


# ─────────────────────────────────────────────────────────────────────────────
# 4.  Base config  (all variants inherit from this, only shared_dim changes)
# ─────────────────────────────────────────────────────────────────────────────

cfg_base = {
    'drug_node_feat_dim': total_atom_dim,
    'drug_edge_feat_dim': total_bond_dim,
    'drug_hidden_dim':    128,
    'drug_num_layers':    3,
    'drug_heads':         4,
    'protein_feat_dim':   protein_features_tensor.shape[1],
    'effect_feat_dim':    effect_features_tensor.shape[1],
    'shared_dim':         256,          # overridden per Track-2 variant
    'num_hetero_layers':  3,
}

drug_batch_full = Batch.from_data_list(drug_pyg_indexed)

# ── Verify actual parameter counts ───────────────────────────────────────────
full_params = sum(p.numel() for p in PharmacologyHeteroGNN(cfg_base).parameters())

print(f"\nDevice: {device}")
print(f"\nTrack 2 — Parameter counts (verified against actual feature dims):")
print(f"  {'shared_dim':>12} {'Parameters':>14} {'% of sd=256':>13}  Label")
print(f"  {'─'*55}")
for sd in [64, 128, 192, 256]:
    cfg_tmp = {**cfg_base, 'shared_dim': sd}
    n = sum(p.numel() for p in PharmacologyHeteroGNN(cfg_tmp).parameters())
    pct = n / full_params * 100
    lbl = '← full model (baseline)' if sd == 256 else ''
    print(f"  {sd:>12} {n:>14,} {pct:>12.1f}%  {lbl}")

print(f"\nTrack 1 — Data fractions (shared_dim=256, {full_params:,} params):")
n_dp = dp_train_edge_index.shape[1]
n_di = di_train_edge_index.shape[1]
for frac in [0.25, 0.50, 0.75, 1.00]:
    dp_k = int(n_dp * frac)
    di_k = int(n_di * frac)
    lbl = '← full model (baseline)' if frac == 1.0 else ''
    print(f"  {frac*100:>5.0f}%  DP train edges: {dp_k:>6,}   DI train edges: {di_k:>6,}  {lbl}")

print(f"\n✓ PharmacologyHeteroGNN ready")


SCALING ABLATION STUDY  —  PharmacologyHeteroGNN
Track 1: Data Scaling    (shared_dim=256, vary training data fraction)
Track 2: Parameter Scaling (100% data, vary shared_dim)

Device: cuda

Track 2 — Parameter counts (verified against actual feature dims):
    shared_dim     Parameters   % of sd=256  Label
  ───────────────────────────────────────────────────────
            64      1,116,610         32.5%  
           128      1,661,954         48.3%  
           192      2,436,674         70.8%  
           256      3,440,770        100.0%  ← full model (baseline)

Track 1 — Data fractions (shared_dim=256, 3,440,770 params):
     25%  DP train edges:  2,602   DI train edges:  1,417  
     50%  DP train edges:  5,204   DI train edges:  2,834  
     75%  DP train edges:  7,806   DI train edges:  4,251  
    100%  DP train edges: 10,409   DI train edges:  5,668  ← full model (baseline)

✓ PharmacologyHeteroGNN ready


In [26]:
import time
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

print("="*80)
print("SCALING ABLATION — TRAINING RUNNER  (6 experiments)")
print("="*80)

# ── Shared hyperparameters ────────────────────────────────────────────────────
NUM_EPOCHS       = 2000
LR               = 3e-4
WEIGHT_DECAY     = 1e-5
NEG_RATIO        = 1
VAL_EVERY        = 10
PATIENCE         = 30
NEG_PER_POS_EVAL = 50

# ── Static device tensors ─────────────────────────────────────────────────────
_drug_batch_dev = drug_batch_full.to(device)
_prot_feat_dev  = protein_features_tensor.to(device)
_eff_feat_dev   = effect_features_tensor.to(device)
_train_ei_dev   = {k: v.to(device) for k, v in train_edge_index_dict.items()}
_val_ei_dev     = {k: v.to(device) for k, v in val_edge_index_dict.items()}
_test_ei_dev    = {k: v.to(device) for k, v in test_edge_index_dict.items()}
_dp_val_dev     = dp_val_edge_index.to(device)
_di_val_dev     = di_val_edge_index.to(device)
_dp_test_dev    = dp_test_edge_index.to(device)
_di_test_dev    = di_test_edge_index.to(device)

_dp_train_full  = dp_train_edge_index          # kept on CPU; sliced per experiment
_di_train_full  = di_train_edge_index

# ── Evaluation function ───────────────────────────────────────────────────────

@torch.no_grad()
def evaluate_hgnn(mdl, pos_edge_index, relation_type, all_positive_set, x_dict,
                  neg_per_pos=NEG_PER_POS_EVAL):
    mdl.eval()
    if relation_type == 'dp':
        n_tails, head_embs, tail_embs, score_fn = (
            num_proteins, x_dict['drug'], x_dict['protein'], mdl.score_dp)
    else:
        n_tails, head_embs, tail_embs, score_fn = (
            num_effects, x_dict['drug'], x_dict['effect'], mdl.score_di)

    head_to_pos = {}
    for (h, t) in all_positive_set:
        head_to_pos.setdefault(h, set()).add(t)

    n_pos = pos_edge_index.shape[1]
    reciprocal_ranks, hits_at = [], {1: 0, 3: 0, 10: 0}
    auc_y_true, auc_y_score   = [], []
    EVAL_BATCH, TAIL_CHUNK    = 64, 512

    for start in range(0, n_pos, EVAL_BATCH):
        end        = min(start + EVAL_BATCH, n_pos)
        heads      = pos_edge_index[0, start:end]
        true_tails = pos_edge_index[1, start:end]
        B          = end - start
        h_emb      = head_embs[heads]

        all_scores = torch.zeros(B, n_tails, device=device)
        for t0 in range(0, n_tails, TAIL_CHUNK):
            t1    = min(t0 + TAIL_CHUNK, n_tails)
            T     = t1 - t0
            t_emb = tail_embs[t0:t1]
            h_exp = h_emb.unsqueeze(1).expand(B, T, -1).reshape(B * T, -1)
            t_exp = t_emb.unsqueeze(0).expand(B, T, -1).reshape(B * T, -1)
            all_scores[:, t0:t1] = score_fn(h_exp, t_exp).reshape(B, T)

        scores_np = all_scores.cpu().numpy()
        for i in range(B):
            h, true_t     = heads[i].item(), true_tails[i].item()
            true_score    = scores_np[i, true_t]
            known         = head_to_pos.get(h, set())
            filt          = scores_np[i].copy()
            for kt in known:
                if kt != true_t and kt < n_tails:
                    filt[kt] = -np.inf
            rank = 1 + int((filt > true_score).sum())
            reciprocal_ranks.append(1.0 / rank)
            for k in hits_at:
                if rank <= k:
                    hits_at[k] += 1
            neg_pool = [t for t in range(n_tails) if t != true_t and t not in known]
            neg_idx  = np.random.choice(neg_pool, min(neg_per_pos, len(neg_pool)), replace=False)
            auc_y_true.append(1.0);  auc_y_score.append(true_score)
            for ns in scores_np[i, neg_idx]:
                auc_y_true.append(0.0); auc_y_score.append(float(ns))

    return {
        'mrr':     float(np.mean(reciprocal_ranks)),
        'hits@1':  hits_at[1]  / n_pos,
        'hits@3':  hits_at[3]  / n_pos,
        'hits@10': hits_at[10] / n_pos,
        'roc_auc': roc_auc_score(np.array(auc_y_true), np.array(auc_y_score)),
        'pr_auc':  average_precision_score(np.array(auc_y_true), np.array(auc_y_score)),
    }

def _get_peak_vram():
    return torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0.0

def _reset_vram():
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()

# ── Experiment configurations ─────────────────────────────────────────────────
#
#   Track 1: shared_dim=256 (full model),  data_frac in {0.25, 0.50, 0.75}
#   Track 2: data_frac=1.00 (full data),   shared_dim in {64, 128, 192}
#   (The 100% data / sd=256 point is the baseline — train separately & fill in)

EXPERIMENTS = [
    # ── Track 1: Data Scaling ──────────────────────────────────────────────
    {'track': 1, 'label': 'Track1 — 25% data,  sd=256', 'shared_dim': 256, 'data_frac': 0.25, 'ckpt': 'scale_data_25.pt'},
    {'track': 1, 'label': 'Track1 — 50% data,  sd=256', 'shared_dim': 256, 'data_frac': 0.50, 'ckpt': 'scale_data_50.pt'},
    {'track': 1, 'label': 'Track1 — 75% data,  sd=256', 'shared_dim': 256, 'data_frac': 0.75, 'ckpt': 'scale_data_75.pt'},
    # ── Track 2: Parameter Scaling ────────────────────────────────────────
    {'track': 2, 'label': 'Track2 — 100% data, sd=64',  'shared_dim':  64, 'data_frac': 1.00, 'ckpt': 'scale_dim_64.pt'},
    {'track': 2, 'label': 'Track2 — 100% data, sd=128', 'shared_dim': 128, 'data_frac': 1.00, 'ckpt': 'scale_dim_128.pt'},
    {'track': 2, 'label': 'Track2 — 100% data, sd=192', 'shared_dim': 192, 'data_frac': 1.00, 'ckpt': 'scale_dim_192.pt'},
]

scaling_results = {}   # key = experiment label

# ── Main experiment loop ──────────────────────────────────────────────────────
for exp in EXPERIMENTS:
    print(f"\n{'='*80}")
    print(f"  {exp['label']}")
    print(f"{'='*80}")

    # ── Build model ──────────────────────────────────────────────────────────
    cfg = {**cfg_base, 'shared_dim': exp['shared_dim']}
    mdl = PharmacologyHeteroGNN(cfg).to(device)
    total_params = sum(p.numel() for p in mdl.parameters())
    full_params  = sum(p.numel() for p in PharmacologyHeteroGNN({**cfg_base, 'shared_dim': 256}).parameters())
    print(f"  Parameters: {total_params:,}  ({total_params/full_params*100:.1f}% of sd=256 model)")

    # ── Subset training edges for Track 1 ────────────────────────────────────
    frac = exp['data_frac']
    n_dp_full = _dp_train_full.shape[1]
    n_di_full = _di_train_full.shape[1]

    if frac < 1.0:
        torch.manual_seed(42)       # reproducible subsets
        perm_dp  = torch.randperm(n_dp_full)[:int(n_dp_full * frac)]
        perm_di  = torch.randperm(n_di_full)[:int(n_di_full * frac)]
        dp_train_pos = _dp_train_full[:, perm_dp].to(device)
        di_train_pos = _di_train_full[:, perm_di].to(device)
    else:
        dp_train_pos = _dp_train_full.to(device)
        di_train_pos = _di_train_full.to(device)

    n_dp_train = dp_train_pos.shape[1]
    n_di_train = di_train_pos.shape[1]
    print(f"  DP train edges: {n_dp_train:,}   DI train edges: {n_di_train:,}\n")

    # ── Negative samplers (same pool, proportional count) ──────────────────
    def _neg_dp():
        negs = sample_negatives_dp_dynamic(
            num_samples    = n_dp_train * NEG_RATIO,
            verified_negs  = verified_dp_train,
            existing_edges = existing_dp,
            valid_srcs     = train_drugs_with_pos,
            valid_tgts     = train_proteins_with_pos,
        )
        return torch.tensor(negs, dtype=torch.long).t().to(device)

    def _neg_di():
        negs = sample_negatives_di_dynamic(
            num_samples    = n_di_train * NEG_RATIO,
            hard_negs      = hard_neg_edges,
            med_negs       = med_neg_edges,
            existing_edges = existing_di,
            valid_srcs     = train_drugs_di,
            valid_tgts     = train_effects_di,
        )
        return torch.tensor(negs, dtype=torch.long).t().to(device)

    # ── Training loop ─────────────────────────────────────────────────────────
    optimizer        = torch.optim.Adam(mdl.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    best_val_mrr     = 0.0
    patience_counter = 0
    total_train_time = 0.0

    for epoch in range(1, NUM_EPOCHS + 1):
        _reset_vram()
        t0 = time.time()
        mdl.train()

        x_dict   = mdl.encode(_drug_batch_dev, _prot_feat_dev, _eff_feat_dev, _train_ei_dev)
        dp_neg   = _neg_dp()
        di_neg   = _neg_di()

        dp_pos   = mdl.score_dp(x_dict['drug'][dp_train_pos[0]], x_dict['protein'][dp_train_pos[1]])
        dp_neg_s = mdl.score_dp(x_dict['drug'][dp_neg[0]],       x_dict['protein'][dp_neg[1]])
        loss_dp  = PharmacologyHeteroGNN.margin_loss(dp_pos, dp_neg_s)

        di_pos   = mdl.score_di(x_dict['drug'][di_train_pos[0]], x_dict['effect'][di_train_pos[1]])
        di_neg_s = mdl.score_di(x_dict['drug'][di_neg[0]],       x_dict['effect'][di_neg[1]])
        loss_di  = PharmacologyHeteroGNN.margin_loss(di_pos, di_neg_s)

        loss = loss_dp + loss_di
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_time += time.time() - t0

        # ── Validation ────────────────────────────────────────────────────────
        if epoch % VAL_EVERY == 0 or epoch == 1:
            mdl.eval()
            with torch.no_grad():
                x_val = mdl.encode(_drug_batch_dev, _prot_feat_dev, _eff_feat_dev, _val_ei_dev)
            val_dp  = evaluate_hgnn(mdl, _dp_val_dev, 'dp', existing_dp, x_val)
            val_di  = evaluate_hgnn(mdl, _di_val_dev, 'di', existing_di, x_val)
            avg_mrr = (val_dp['mrr'] + val_di['mrr']) / 2

            print(f"  Ep {epoch:>4d}/{NUM_EPOCHS}  "
                  f"Loss:{loss.item():.4f} (DP={loss_dp.item():.4f} DI={loss_di.item():.4f})  "
                  f"VRAM:{_get_peak_vram():.0f}MB")
            print(f"    Val DP — MRR:{val_dp['mrr']:.4f}  H@10:{val_dp['hits@10']:.4f}  "
                  f"ROC-AUC:{val_dp['roc_auc']:.4f}  PR-AUC:{val_dp['pr_auc']:.4f}")
            print(f"    Val DI — MRR:{val_di['mrr']:.4f}  H@10:{val_di['hits@10']:.4f}  "
                  f"ROC-AUC:{val_di['roc_auc']:.4f}  PR-AUC:{val_di['pr_auc']:.4f}")

            if avg_mrr > best_val_mrr:
                best_val_mrr     = avg_mrr
                patience_counter = 0
                torch.save(mdl.state_dict(), exp['ckpt'])
                print(f"    ★ New best avg MRR: {avg_mrr:.4f} — checkpoint saved")
            else:
                patience_counter += 1
                if patience_counter >= PATIENCE:
                    print(f"\n  Early stopping at epoch {epoch}")
                    break

    # ── Test evaluation ────────────────────────────────────────────────────────
    mdl.load_state_dict(torch.load(exp['ckpt'], map_location=device, weights_only=True))
    mdl.eval()
    with torch.no_grad():
        x_test = mdl.encode(_drug_batch_dev, _prot_feat_dev, _eff_feat_dev, _test_ei_dev)
    test_dp = evaluate_hgnn(mdl, _dp_test_dev, 'dp', existing_dp, x_test)
    test_di = evaluate_hgnn(mdl, _di_test_dev, 'di', existing_di, x_test)

    print(f"\n  ── Test results — {exp['label']} ──")
    print(f"  Drug-Protein: MRR={test_dp['mrr']:.4f}  H@10={test_dp['hits@10']:.4f}  "
          f"ROC-AUC={test_dp['roc_auc']:.4f}  PR-AUC={test_dp['pr_auc']:.4f}")
    print(f"  Drug-Effect:  MRR={test_di['mrr']:.4f}  H@10={test_di['hits@10']:.4f}  "
          f"ROC-AUC={test_di['roc_auc']:.4f}  PR-AUC={test_di['pr_auc']:.4f}")

    scaling_results[exp['label']] = {
        'track':        exp['track'],
        'shared_dim':   exp['shared_dim'],
        'data_frac':    exp['data_frac'],
        'params':       total_params,
        'train_time_s': total_train_time,
        'best_val_mrr': best_val_mrr,
        'dp':           test_dp,
        'di':           test_di,
    }

    del mdl
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f"\n{'='*80}")
print("ALL 6 EXPERIMENTS COMPLETE — run next cell for comparison tables")
print(f"{'='*80}")


SCALING ABLATION — TRAINING RUNNER  (6 experiments)



  Track1 — 25% data,  sd=256
  Parameters: 3,440,770  (100.0% of sd=256 model)
  DP train edges: 2,602   DI train edges: 1,417

  Ep    1/2000  Loss:2.0151 (DP=1.0164 DI=0.9987)  VRAM:5226MB
    Val DP — MRR:0.0159  H@10:0.0316  ROC-AUC:0.6434  PR-AUC:0.0393
    Val DI — MRR:0.0110  H@10:0.0240  ROC-AUC:0.5276  PR-AUC:0.0223
    ★ New best avg MRR: 0.0135 — checkpoint saved
  Ep   10/2000  Loss:1.2935 (DP=0.6971 DI=0.5965)  VRAM:5258MB
    Val DP — MRR:0.0202  H@10:0.0542  ROC-AUC:0.7338  PR-AUC:0.0625
    Val DI — MRR:0.0761  H@10:0.1003  ROC-AUC:0.6256  PR-AUC:0.0341
    ★ New best avg MRR: 0.0482 — checkpoint saved
  Ep   20/2000  Loss:0.8973 (DP=0.4786 DI=0.4186)  VRAM:5258MB
    Val DP — MRR:0.0182  H@10:0.0679  ROC-AUC:0.7681  PR-AUC:0.0508
    Val DI — MRR:0.0823  H@10:0.1412  ROC-AUC:0.6683  PR-AUC:0.0428
    ★ New best avg MRR: 0.0503 — checkpoint saved
  Ep   30/2000  Loss:0.6901 (DP=0.3645 DI=0.3256)  VRAM:5257MB
    Val DP — MRR:0.0175  H@10:0.0373  ROC-AUC:0.8263  PR-AUC:

In [27]:
print("="*80)
print("SCALING ABLATION — COMPARISON TABLES")
print("="*80)

METRICS       = ['roc_auc', 'pr_auc', 'mrr', 'hits@1', 'hits@3', 'hits@10']
METRIC_LABELS = ['ROC-AUC', 'PR-AUC',  'MRR',  'H@1',   'H@3',   'H@10']

# ── Fill in full-model baseline results from "3 Million Paramater New Eval Model.ipynb"
BASELINE = {
    'label':      'Baseline — 100% data, sd=256  (3.1M params)',
    'params':     None,    # e.g. 3_115_010
    'data_frac':  1.00,
    'shared_dim': 256,
    'dp': {'roc_auc': None, 'pr_auc': None, 'mrr': None,
           'hits@1': None, 'hits@3': None, 'hits@10': None},
    'di': {'roc_auc': None, 'pr_auc': None, 'mrr': None,
           'hits@1': None, 'hits@3': None, 'hits@10': None},
}

def _fmt(v):
    return f"{v:>9.4f}" if v is not None else f"{'—':>9}"

def _print_track(track_num, track_label, rows, sort_key):
    """rows = list of (label, result_dict), sort_key = 'data_frac' or 'shared_dim'"""
    rows_sorted = sorted(rows, key=lambda r: r[1][sort_key])
    for rel, rel_label in [('dp', 'Drug–Protein (binds_to)'), ('di', 'Drug–Effect (treats)')]:
        print(f"\n{'─'*90}")
        print(f"  Track {track_num}: {track_label}  —  {rel_label}")
        print(f"{'─'*90}")
        hdr = f"  {'Experiment':<40}" + "".join(f"{lbl:>9}" for lbl in METRIC_LABELS) + f"  {'Params':>12}"
        print(hdr)
        print(f"  {'─'*39} " + " ".join(["─"*8] * len(METRICS)) + f"  {'─'*12}")
        # Baseline row first
        if BASELINE[rel]['roc_auc'] is not None:
            vals = "".join(_fmt(BASELINE[rel][m]) for m in METRICS)
            p = f"{BASELINE['params']:,}" if BASELINE['params'] else "—"
            print(f"  {BASELINE['label']:<40}{vals}  {p:>12}")
        for lbl, res in rows_sorted:
            vals = "".join(_fmt(res['di' if rel == 'di' else 'dp'][m]) for m in METRICS)
            p    = f"{res['params']:,}"
            print(f"  {lbl:<40}{vals}  {p:>12}")

# ── Separate the two tracks ───────────────────────────────────────────────────
track1_rows = [(lbl, res) for lbl, res in scaling_results.items() if res['track'] == 1]
track2_rows = [(lbl, res) for lbl, res in scaling_results.items() if res['track'] == 2]

_print_track(1, 'Data Scaling    (sd=256 fixed)',  track1_rows,  'data_frac')
_print_track(2, 'Param Scaling   (100% data fixed)', track2_rows, 'shared_dim')

# ── Delta tables vs baseline ──────────────────────────────────────────────────
if BASELINE['dp']['roc_auc'] is not None:
    print(f"\n{'='*90}")
    print("  Δ vs Baseline  (negative = this variant is WORSE than the full baseline)")
    print(f"{'='*90}")
    for track_num, track_rows, sort_key, track_label in [
        (1, track1_rows, 'data_frac',  'Data Scaling'),
        (2, track2_rows, 'shared_dim', 'Param Scaling'),
    ]:
        for rel, rel_label in [('dp', 'Drug–Protein'), ('di', 'Drug–Effect')]:
            print(f"\n  Track {track_num}: {track_label}  —  {rel_label}")
            hdr = f"  {'Experiment':<40}" + "".join(f"{lbl:>9}" for lbl in METRIC_LABELS)
            print(hdr)
            print(f"  {'─'*39} " + " ".join(["─"*8] * len(METRICS)))
            for lbl, res in sorted(track_rows, key=lambda r: r[1][sort_key]):
                deltas = "".join(
                    f"{(res[rel][m] - BASELINE[rel][m]):>+9.4f}" for m in METRICS
                )
                print(f"  {lbl:<40}{deltas}")

# ── Summary: learning rate per unit of resource ───────────────────────────────
print(f"\n{'='*90}")
print("  Resource Efficiency Summary")
print(f"{'='*90}")
print(f"  {'Experiment':<40}  {'data_frac':>10}  {'shared_dim':>10}  "
      f"{'Params':>10}  {'TrainTime':>10}  {'BestValMRR':>11}")
print(f"  {'─'*39}  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*11}")
all_rows_sorted = sorted(
    scaling_results.items(),
    key=lambda kv: (kv[1]['track'], kv[1]['data_frac'], kv[1]['shared_dim'])
)
for lbl, res in all_rows_sorted:
    t = f"{res['train_time_s']/60:.1f}m"
    print(f"  {lbl:<40}  {res['data_frac']:>10.0%}  {res['shared_dim']:>10}  "
          f"  {res['params']:>9,}  {t:>10}  {res['best_val_mrr']:>11.4f}")

# ── Interpretation guide ──────────────────────────────────────────────────────
print(f"\n{'='*90}")
print("  How to read these results")
print(f"{'='*90}")
print("  Track 1 (data scaling):  If performance drops sharply from 100%→75%→50%→25%,")
print("                           MORE DATA is the limiting factor for this model.")
print("  Track 2 (param scaling): If performance is flat from sd=64→128→192→256,")
print("                           MODEL SIZE is NOT the bottleneck — data quality/quantity is.")
print("  Key question:  Is the gap 'Baseline − 25% data' larger or smaller than")
print("                 'Baseline − sd=64'?  The bigger gap reveals the true constraint.")


SCALING ABLATION — COMPARISON TABLES

──────────────────────────────────────────────────────────────────────────────────────────
  Track 1: Data Scaling    (sd=256 fixed)  —  Drug–Protein (binds_to)
──────────────────────────────────────────────────────────────────────────────────────────
  Experiment                                ROC-AUC   PR-AUC      MRR      H@1      H@3     H@10        Params
  ─────────────────────────────────────── ──────── ──────── ──────── ──────── ──────── ────────  ────────────
  Track1 — 25% data,  sd=256                 0.9514   0.4320   0.1734   0.0789   0.1841   0.3782     3,440,770
  Track1 — 50% data,  sd=256                 0.9633   0.5140   0.2281   0.1205   0.2520   0.4603     3,440,770
  Track1 — 75% data,  sd=256                 0.9682   0.5381   0.2444   0.1310   0.2751   0.4850     3,440,770

──────────────────────────────────────────────────────────────────────────────────────────
  Track 1: Data Scaling    (sd=256 fixed)  —  Drug–Effect (treat